# 第 64 课｜PPO 训练稳定性与可复现实验协议

**从单条训练曲线，走向多随机种子、开发集选择与留出评测。**

前面三课分别完成价值目标、Actor/Critic联合训练、minibatch与梯度累积。本课沿用同一个三状态、两动作的教学 PPO 实现，关注实验是否可信，而不是为某次最好看的结果寻找理由。

本课交付两个**互不依赖**的文件：自包含 Jupyter 讲义与完整 Python 脚本。Python 3.10+，已有 PyTorch 即可；默认单线程 CPU、float64，无模型下载和联网。

**研究边界：** 两种时间上限（5步与7步），属于同一玩具环境的两个任务配置，不是大型独立任务集；开发3个训练seed，留出4个训练seed。百分位bootstrap区间是描述性不确定性检查，不保证这样少的种子可以得出稳健统计显著性结论；不报告预训练模型能力或3060性能。

## 0. 你将完成什么？

1. **锁定计划：** 任务、候选学习率、开发训练种子、留出训练种子、统一训练预算与选择规则。
2. **完整开发实验：** 每个候选在每个任务与种子上从头训练，按任务等权的开发平均指标选择超参数。
3. **锁定选择，再访问留出种子：** 仅比较事先声明的基线和开发选中候选，不在测试上重新挑参数。
4. **配对统计：** 同一任务、同一训练seed两条配置成对相减；在**训练seed层**重采样，而不是把评估回合数当作独立训练次数。
5. **报告预算与失败：** 保留每次训练结果、优化器次数、环境transition、完整轨迹和检查点，不删除失败的种子。

默认执行完整实验；若只想检查代码，可以在后文设 `RUN_FULL_EXPERIMENT=False`，或在终端先运行 `--checks-only`、`--math-only`。后文有独立练习、参考实现和固定检查。

### 实现：约定、类型检查、稳定种子与 Config

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第64课：PPO训练稳定性与可复现实验协议。

Python 3.10+、PyTorch。沿用内置三状态/两动作PPO教学Actor-Critic，
先冻结任务/候选/开发和测试训练种子，再按多个独立训练种子重复实验。
只用开发训练种子选超参数；测试训练种子用于一次性留出比较。
配对按训练种子聚类，任务内成对Bootstrap；不将评估episodes当成独立训练run。
这是CPU小任务研究协议，不是预训练LLM或真实大规模RL显著性证明。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需已有PyTorch环境；请使用匹配的Jupyter内核") from exc

VERSION = "lesson64-multiseed-protocol-v1"
SOURCES = {
    "ppo_paper": "https://arxiv.org/abs/1707.06347",
    "spinningup_ppo": "https://spinningup.openai.com/en/latest/algorithms/ppo.html",
    "spinningup_code": "https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html",
    "sb3_code": "https://stable-baselines3.readthedocs.io/en/v2.7.1/_modules/stable_baselines3/ppo/ppo.html",
    "gae": "https://arxiv.org/abs/1506.02438",
    "gymnasium_time_limit": "https://gymnasium.farama.org/v0.26.3/tutorials/handling_time_limits/",
    "pytorch_autograd": "https://docs.pytorch.org/docs/stable/notes/autograd.html",
    "statistical_precipice": "https://arxiv.org/abs/2108.13264",
    "rliable": "https://github.com/google-research/rliable",
    "deep_rl_matters": "https://ojs.aaai.org/index.php/AAAI/article/view/11694",
    "sb3_rl_tips": "https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html",
}

class LessonError(ValueError):
    """本课程协议错误；不静默修改任务定义。"""

def require(condition, message):
    if not bool(condition):
        raise LessonError(message)

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def sha256(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def tensor_hash(named):
    h = hashlib.sha256()
    for name, tensor in sorted(named.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()

def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)

@dataclass(frozen=True)
class Config:
    seed: int = 6301
    gamma: float = .95
    gae_lambda: float = .95
    horizon: int = 6
    rollouts: int = 24
    episodes_per_rollout: int = 25
    epochs: int = 3
    minibatch_size: int = 17
    microbatch_size: int = 5
    actor_lr: float = .012
    critic_lr: float = .010
    actor_hidden: int = 24
    critic_hidden: int = 24
    clip_ratio: float = .2
    value_clip: float = .2
    entropy_coef: float = .006
    target_kl: float = .06
    grad_clip: float = 1.
    eval_episodes: int = 120

    def validate(self):
        for name in ('seed','horizon','rollouts','episodes_per_rollout','epochs',
                     'minibatch_size','microbatch_size','actor_hidden','critic_hidden','eval_episodes'):
            v=getattr(self,name)
            require(type(v) is int and v >= (0 if name=='seed' else 1),name+'必须为整数')
        require(3<=self.horizon<=16 and 1<=self.epochs<=8, 'horizon/epochs超出教学范围')
        require(2<=self.minibatch_size<=512 and 1<=self.microbatch_size<=self.minibatch_size,
                'microbatch必须小于等于minibatch')
        for name in ('gamma','gae_lambda','actor_lr','critic_lr','clip_ratio','value_clip',
                     'entropy_coef','target_kl','grad_clip'):
            v=getattr(self,name)
            require(type(v) in (float,int) and math.isfinite(v),name+'必须有限数值')
        require(0<self.gamma<1 and 0<=self.gae_lambda<=1,'gamma或lambda非法')
        require(self.actor_lr>0 and self.critic_lr>0 and 0<self.clip_ratio<1
                and self.value_clip>0 and self.entropy_coef>=0 and self.target_kl>0
                and self.grad_clip>0,'训练/裁剪参数非法')

In [2]:
torch.set_num_threads(1)
print("Python:",platform.python_version())
print("PyTorch:",torch.__version__)
print("本课设备：CPU，dtype=float64，线程数=1")

Python: 3.13.5
PyTorch: 2.10.0+cpu
本课设备：CPU，dtype=float64，线程数=1


## 1. 实验单位是什么？

同一个训练好的策略上评估100个回合，得到的是**一份训练run的评估样本**，不是100次独立模型训练。

三层单位需要分清：

| 层次 | 本课例子 | 用途 |
|---|---|---|
| 环境评估回合 | 某个模型上96个回合 | 测量固定策略的采样波动 |
| 独立训练seed | 从随机初始化开始训练一次 | 比较训练过程的随机性 |
| 任务配置 | 5步或7步的同环境 | 观察条件差异，不等于独立领域 |

本课有有限状态，能根据三个状态的动作概率**精确计算给定步数内到达目标的概率**。这作为主要评测，避免把评估采样噪声误认为训练改进；同时保留真实采样回合的结果，用来观察测量差异。

一般模型、随机环境和开放式任务不一定能拥有这个精确评估条件。

### 实现：受控环境、终止/截断、GAE及精确有限时域评估

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [3]:
GOAL=3
ACTION_WAIT, ACTION_ADVANCE=0,1
STEP_COST,SUCCESS_BONUS=-.01,1.0


@dataclass(frozen=True)
class Step:
    obs: int
    action: int
    reward: float
    next_obs: int
    terminated: bool
    truncated: bool
    old_logp: float

def validate_step(step):
    require(isinstance(step, Step), "Step类型错误")
    require(type(step.obs) is int and 0 <= step.obs < GOAL, "当前obs必须是非终止位置")
    require(type(step.next_obs) is int and 0 <= step.next_obs <= GOAL, "next_obs无效")
    require(type(step.action) is int and step.action in (ACTION_WAIT, ACTION_ADVANCE), "动作非法")
    require(type(step.terminated) is bool and type(step.truncated) is bool,
            "terminated/truncated必须是bool")
    require(not (step.terminated and step.truncated), "真实终止与外部截断在本课程中互斥")
    require(step.next_obs == min(step.obs + step.action, GOAL), "环境动态不匹配")
    require(step.terminated == (step.next_obs == GOAL), "终止状态不匹配")
    expected_reward = STEP_COST + (SUCCESS_BONUS if step.terminated else 0)
    require(math.isfinite(step.reward) and abs(step.reward - expected_reward) < 1e-10,
            "奖励必须来自固定环境，而不是模型自报")
    require(type(step.old_logp) is float and math.isfinite(step.old_logp) and step.old_logp <= 1e-6,
            "需要有效的行为log概率")

def validate_episode(episode, horizon):
    require(type(episode) is list and 1 <= len(episode) <= horizon, "episode长度错误")
    require(episode[0].obs == 0, "每回合必须从位置0开始")
    for i, step in enumerate(episode):
        validate_step(step)
        if i < len(episode) - 1:
            require(not step.terminated and not step.truncated, "回合结束后仍有步骤")
            require(episode[i + 1].obs == step.next_obs, "下一步观察不连续")
    require(episode[-1].terminated or episode[-1].truncated, "回合尾部必须有明确终止状态")
    require(not episode[-1].truncated or len(episode) == horizon,
            "外部时间上限只能在约定horizon触发")

def transition(pos, action, t, horizon):
    require(type(pos) is int and 0 <= pos < GOAL, "pos非法")
    require(type(action) is int and action in (0, 1), "action非法")
    require(type(t) is int and 0 <= t < horizon, "时间步非法")
    nxt = min(pos + action, GOAL)
    terminated = nxt == GOAL
    truncated = t + 1 >= horizon and not terminated
    reward = STEP_COST + (SUCCESS_BONUS if terminated else 0.)
    return nxt, reward, terminated, truncated

def checked_numeric(values, name):
    require(isinstance(values, (list, tuple)) and len(values) > 0, name + "必须为非空序列")
    require(all(type(x) in (int, float) and math.isfinite(x) for x in values), name + "必须有限")

def discounted_returns(rewards, gamma, bootstrap=0.0):
    """时间倒序累计，最后一个非终止cut使用V(final_obs)作为尾部。"""
    checked_numeric(rewards, "rewards")
    require(type(gamma) in (int, float) and 0 < gamma <= 1, "gamma非法")
    require(type(bootstrap) in (int, float) and math.isfinite(bootstrap), "bootstrap非法")
    acc = float(bootstrap)
    output = [0.] * len(rewards)
    for i in range(len(rewards)-1, -1, -1):
        acc = float(rewards[i]) + gamma*acc
        output[i] = acc
    return output

def generalized_advantage_estimation(episodes, values, next_values, gamma, lam):
    """独立episode：delta=r+gamma*(1-terminated)V(next)-V(s)。

    截断时delta保留V(final_obs)，但由于下一条记录是新回合，trace递推停止。
    自然rollout数组末尾也停止trace，若未终止则必须保留下一状态bootstrap。
    返回 A 与 lambda-return=A+V_old，不连接critic梯度。
    """
    require(type(episodes) is list and bool(episodes), "需要回合列表")
    require(type(gamma) in (float, int) and 0 < gamma <= 1, "gamma非法")
    require(type(lam) in (float, int) and 0 <= lam <= 1, "lambda非法")
    count = sum(len(ep) for ep in episodes)
    checked_numeric(values, "values")
    checked_numeric(next_values, "next_values")
    require(len(values) == len(next_values) == count, "值函数数组长度与步骤数量不符")
    advantages, targets, deltas = [], [], []
    offset = 0
    for ep in episodes:
        require(len(ep) > 0, "空回合")
        for j, step in enumerate(ep):
            validate_step(step)
            if j < len(ep)-1:
                require(not (step.terminated or step.truncated), "episode结束后仍有下一条步骤")
                require(ep[j+1].obs == step.next_obs, "GAE回合内部不连续")
        local_values = values[offset:offset + len(ep)]
        local_next = next_values[offset:offset + len(ep)]
        offset += len(ep)
        local_d = [step.reward + gamma*(0. if step.terminated else nv) - v
                   for step,v,nv in zip(ep,local_values,local_next)]
        local_adv = [0.] * len(ep)
        ahead = 0.
        for j in range(len(ep)-1, -1, -1):
            # 回合结束或采样切断后，不能把下一回合的优势连接过来。
            boundary = ep[j].terminated or ep[j].truncated
            ahead = local_d[j] + gamma*lam*(0. if boundary else ahead)
            local_adv[j] = ahead
        deltas.extend(local_d)
        advantages.extend(local_adv)
        targets.extend([a+v for a,v in zip(local_adv,local_values)])
    return {"delta":deltas, "advantage":advantages, "lambda_target":targets}

def exact_policy_values(forward_probabilities, gamma=0.95):
    """Bellman: 在底层无限时域、位置马尔可夫过程里计算精确V^pi。

    外部6步截断不属于底层MDP，所以不能把它写进该值函数。
    """
    checked_numeric(forward_probabilities, "forward_probabilities")
    require(len(forward_probabilities) == GOAL, "每个非终止位置一个前进概率")
    require(all(0 <= v <= 1 for v in forward_probabilities), "动作概率不在0..1")
    require(type(gamma) in (int,float) and 0 < gamma < 1, "无穷时域要求gamma小于1")
    values = [0.] * GOAL
    for s in range(GOAL-1, -1, -1):
        p = float(forward_probabilities[s])
        forward_value = SUCCESS_BONUS + STEP_COST if s == GOAL-1 else STEP_COST + gamma*values[s+1]
        values[s] = (p*forward_value + (1-p)*STEP_COST)/(1 - gamma*(1-p))
    return values

def finite_horizon_success_probability(forward_probabilities, horizon):
    """固定外部horizon内到达目标的精确概率，不等于底层无限时域V。"""
    require(type(horizon) is int and horizon >= 1, "horizon非法")
    checked_numeric(forward_probabilities, "policy")
    require(len(forward_probabilities)==GOAL and all(0 <= x <= 1 for x in forward_probabilities),
            "策略概率非法")
    active = [1., 0., 0.]
    finished = 0.
    for _ in range(horizon):
        new = [0.] * GOAL
        for s,pop in enumerate(active):
            prob = forward_probabilities[s]
            new[s] += pop*(1-prob)
            if s == GOAL-1:
                finished += pop*prob
            else:
                new[s+1] += pop*prob
        active = new
    require(abs(finished+sum(active)-1)<1e-9, "总概率不归一")
    return finished

In [4]:
print("动作：0=等待，1=前进；到位置3真正终止，达到时间上限则外部截断。")
for h in (5,7):
    p=[.3,.4,.5]
    print("相同策略下 horizon",h,"精确成功概率",finite_horizon_success_probability(p,h))
print("截断保留bootstrapping的价值，不应把后续回合接进GAE。")

动作：0=等待，1=前进；到位置3真正终止，达到时间上限则外部截断。
相同策略下 horizon 5 精确成功概率 0.2982
相同策略下 horizon 7 精确成功概率 0.5494859999999999
截断保留bootstrapping的价值，不应把后续回合接进GAE。


### 实现：3状态 Actor/Critic、可重复rollout及两类评估

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [5]:
class Actor(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,2))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "actor状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "actor状态越界")
        return self.net(states)

class Critic(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,1))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "critic状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "critic状态越界")
        return self.net(states).squeeze(-1)

def actor_probabilities(actor):
    with torch.no_grad():
        actor.eval()
        probs=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)[:,ACTION_ADVANCE]
    return probs.tolist()

def value_prediction(critic):
    with torch.no_grad():
        critic.eval()
        return critic(torch.arange(GOAL,dtype=torch.long)).tolist()

def value_error(critic,actor,gamma):
    reference=exact_policy_values(actor_probabilities(actor),gamma)
    prediction=value_prediction(critic)
    return {"predicted":prediction,"exact_for_current_policy":reference,
            "rmse":math.sqrt(sum((a-b)**2 for a,b in zip(prediction,reference))/GOAL),
            "scope":"exact_infinite_horizon_value_of_current_policy_not_finite_time_limit_return"}

def collect_rollouts(actor, episodes, horizon, rng):
    require(type(episodes) is int and episodes >= 1, "episodes非法")
    require(type(horizon) is int and horizon >= 3, "horizon非法")
    result=[]
    actor.eval()
    with torch.no_grad():
        p=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)
    for _ in range(episodes):
        pos=0;ep=[]
        for t in range(horizon):
            action=1 if rng.random()<float(p[pos,1]) else 0
            logp=math.log(float(p[pos,action]))
            nxt,reward,terminated,truncated=transition(pos,action,t,horizon)
            ep.append(Step(pos,action,reward,nxt,terminated,truncated,logp))
            if terminated or truncated:
                break
            pos=nxt
        validate_episode(ep,horizon)
        result.append(ep)
    return result

def compute_targets(episodes,critic,gamma,lam,bootstrap_truncation=True):
    require(type(bootstrap_truncation) is bool, "bootstrap标志必须为bool")
    states=[s.obs for ep in episodes for s in ep]
    nextstates=[s.next_obs if not s.terminated else 0 for ep in episodes for s in ep]
    with torch.no_grad():
        values=critic(torch.tensor(states,dtype=torch.long)).tolist()
        nextvalues=critic(torch.tensor(nextstates,dtype=torch.long)).tolist()
    for i,step in enumerate(s for ep in episodes for s in ep):
        if step.terminated or (step.truncated and not bootstrap_truncation):
            nextvalues[i]=0.
    target=generalized_advantage_estimation(episodes,values,nextvalues,gamma,lam)
    return {"states":states,"next_states":nextstates,"values_old":values,
            "next_values_old":nextvalues,"advantages":target['advantage'],
            "lambda_returns":target['lambda_target'],"deltas":target['delta'],
            "old_logps":[s.old_logp for ep in episodes for s in ep],
            "actions":[s.action for ep in episodes for s in ep],
            "rewards":[s.reward for ep in episodes for s in ep],
            "terminated":[s.terminated for ep in episodes for s in ep],
            "truncated":[s.truncated for ep in episodes for s in ep],
            "episodes":len(episodes),"steps":len(states),
            "bootstrap_truncation":bootstrap_truncation,
            "scope":"on_policy_fixed_targets_detached_from_critic"}

def evaluate_policy(actor,critic,gamma,horizon,rollouts=200,seed=2026):
    prob=actor_probabilities(actor)
    exact_continuing=exact_policy_values(prob,gamma)
    exact_success=finite_horizon_success_probability(prob,horizon)
    sampled=collect_rollouts(actor,rollouts,horizon,random.Random(seed))
    success=sum(ep[-1].terminated for ep in sampled)
    truncated=sum(ep[-1].truncated for ep in sampled)
    return {"forward_probabilities":prob,"exact_continuing_value":exact_continuing,
            "exact_success_within_limit":exact_success,
            "sampled_successes":success,"sampled_episodes":rollouts,
            "sampled_success_rate":success/rollouts,"sampled_truncations":truncated,
            "value_rmse":value_error(critic,actor,gamma)['rmse'],
            "note":"固定环境的精确计算＋有限采样；并非模型数学推理能力"}

## 2. 只改变一个超参数，保持其他设置相同

本课预声明三个候选：Actor 学习率 `0.004`、`0.012`、`0.030`。Critic 学习率、epoch、minibatch、GAE、裁剪、KL停止等全部一致。预登记基线为 `lr_low`。

开发训练seed：6401、6402、6403。留出训练seed：6501、6502、6503、6504。**它们是两组不同的训练重启，不仅仅是评估回合使用不同随机数。**

每个候选和训练seed都会重新初始化 Actor 与 Critic；相同seed的候选共享初始化哈希与外部随机调度种子，便于成对对照。但策略更新改变后，后续动作和实际访问的状态可能不同；共享随机源不保证样本完全一致。

选择规则事先固定：每个任务内先对开发训练seed求均值，再对两个任务等权平均；最高者胜，完全相同时取候选列表中的第一项。不会拿测试种子看完再挑。

### 实现：PPO-Clip、Value-Clip及独立的 KL 诊断

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [6]:
def probability_ratio(new_logp, old_logp):
    require(isinstance(new_logp,torch.Tensor) and isinstance(old_logp,torch.Tensor)
            and new_logp.shape==old_logp.shape and new_logp.ndim==1 and new_logp.numel()>0,
            '新旧log概率需要同形状非空向量')
    require(new_logp.is_floating_point() and old_logp.is_floating_point()
            and new_logp.dtype==old_logp.dtype and new_logp.device==old_logp.device,
            '新旧log概率精度/设备非法')
    require(bool(torch.isfinite(new_logp).all() and torch.isfinite(old_logp).all()),'log概率非有限')
    delta=new_logp-old_logp.detach()
    require(bool((delta.detach().abs()<40).all()),'概率比过大，不静默截断')
    return delta.exp()

def policy_loss(new_logp, old_logp, advantage, epsilon):
    require(type(epsilon) in (int,float) and math.isfinite(epsilon) and 0<epsilon<1,'policy clip范围非法')
    require(isinstance(advantage,torch.Tensor) and advantage.shape==new_logp.shape
            and advantage.dtype==new_logp.dtype and advantage.device==new_logp.device
            and bool(torch.isfinite(advantage).all()),'优势形状/精度/数值非法')
    ratio=probability_ratio(new_logp,old_logp)
    a=advantage.detach()
    plain=ratio*a
    bounded=ratio.clamp(1-epsilon,1+epsilon)*a
    obj=torch.minimum(plain,bounded)
    outside=(ratio>1+epsilon)|(ratio<1-epsilon)
    active=((ratio>1+epsilon)&(a>0))|((ratio<1-epsilon)&(a<0))
    return -obj.mean(),{'ratio':ratio,'surrogate':obj,'plain':plain,
                        'ratio_outside_fraction':float(outside.detach().float().mean()),
                        'active_clip_fraction':float(active.detach().float().mean())}

def value_loss(prediction, old_value, target, epsilon=.2, method='clipmax'):
    """三种可选值损失；clipmax不是原始PPO论文的必选部件。

    plain:        0.5*MSE(pred,target)
    clipmax:      0.5*mean(max(raw_squared, clipped_pred_squared)) (PPO2变体)
    clipped_pred: 0.5*MSE(clipped_prediction,target) (SB3可选实现)
    """
    require(method in {'plain','clipmax','clipped_pred'},'value loss方法不支持')
    require(type(epsilon) in (float,int) and math.isfinite(epsilon) and epsilon>0,'value epsilon无效')
    require(all(isinstance(x,torch.Tensor) for x in (prediction,old_value,target))
            and prediction.ndim==1 and prediction.numel()>0
            and prediction.shape==old_value.shape==target.shape,'value输入必须同形状一维向量')
    require(all(x.is_floating_point() and x.dtype==prediction.dtype and x.device==prediction.device
                and bool(torch.isfinite(x).all()) for x in (prediction,old_value,target)),
            'value设备、dtype或数值非法')
    old=old_value.detach(); goal=target.detach()
    raw_squared=(prediction-goal).square()
    clipped_prediction=old+(prediction-old).clamp(-epsilon,epsilon)
    clipped_squared=(clipped_prediction-goal).square()
    if method=='plain': squared=raw_squared
    elif method=='clipmax': squared=torch.maximum(raw_squared,clipped_squared)
    else: squared=clipped_squared
    return .5*squared.mean(), {
        'raw_half_mse':float((.5*raw_squared.detach()).mean()),
        'clipped_prediction_half_mse':float((.5*clipped_squared.detach()).mean()),
        'max_half_mse':float((.5*torch.maximum(raw_squared.detach(),clipped_squared.detach())).mean()),
        'value_outside_clip_fraction':float(((prediction.detach()-old).abs()>epsilon).float().mean()),
        'method':method}

def normalize_advantages(advantages):
    require(isinstance(advantages,torch.Tensor) and advantages.ndim==1 and advantages.numel()>1
            and advantages.is_floating_point() and bool(torch.isfinite(advantages).all()),
            '归一化优势需要超过一个有限浮点样本')
    a=advantages.detach()
    return (a-a.mean())/(a.std(unbiased=False)+1e-8)

def old_state_kl(old_distribution,new_distribution):
    """仅计算旧样本访问的三状态上的Categorical KL(old||new)的平均。"""
    require(isinstance(old_distribution,torch.Tensor) and isinstance(new_distribution,torch.Tensor)
            and old_distribution.ndim==2 and old_distribution.shape==new_distribution.shape
            and old_distribution.shape[1]==2 and old_distribution.numel()>0,'KL分布形状无效')
    require(all(torch.isfinite(x).all() for x in (old_distribution,new_distribution)),'非有限分布')
    for d in (old_distribution,new_distribution):
        require(bool(torch.allclose(d.logsumexp(-1),torch.zeros_like(d[:,0]),atol=1e-5)),
                'KL需要归一化log概率')
    old=old_distribution.detach()
    return (old.exp()*(old-new_distribution)).sum(-1).mean()

### 实现：冻结旧策略/GAE/return与epoch打乱

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [7]:
def stable_epoch_seed(master: int, rollout_id: int, epoch: int) -> int:
    require(all(type(x) is int and x>=0 for x in (master, rollout_id, epoch)),"种子与索引必须非负整数")
    return int(sha256(["lesson63", master, rollout_id, epoch])[:8],16)


def shuffled_minibatches(step_count: int, minibatch_size: int, seed: int) -> list[list[int]]:
    require(type(step_count) is int and step_count>0,"没有rollout步骤")
    require(type(minibatch_size) is int and minibatch_size>0,"minibatch size非法")
    rng=random.Random(seed)
    ids=list(range(step_count))
    rng.shuffle(ids)
    return [ids[start:start+minibatch_size] for start in range(0,step_count,minibatch_size)]


def split_microbatches(minibatch: list[int], microbatch_size: int) -> list[list[int]]:
    require(type(microbatch_size) is int and microbatch_size>0,"microbatch size非法")
    require(isinstance(minibatch,list) and bool(minibatch),"minibatch为空")
    require(len(set(minibatch))==len(minibatch),"minibatch有重复索引")
    return [minibatch[i:i+microbatch_size] for i in range(0,len(minibatch),microbatch_size)]


def frozen_signature(buffer: dict) -> str:
    names=("states","actions","old_logps","old_values","returns","advantages",
           "raw_advantages","old_distribution","terminated","truncated")
    frozen_tensors=tensor_hash({name:buffer[name] for name in names})
    meta={k:buffer[k] for k in ("rollout_id","episode_lengths","episode_successes",
                              "episode_truncations","sampled_steps")}
    return sha256([frozen_tensors,meta])


def make_buffer(actor: Actor, critic: Critic, episodes: list[list[Step]], cfg: Config,
                rollout_id: int) -> dict:
    cfg.validate()
    require(type(rollout_id) is int and rollout_id>=0,"rollout ID非法")
    require(len(episodes)==cfg.episodes_per_rollout,"episode数量与配置不同")
    for ep in episodes:validate_episode(ep,cfg.horizon)
    values=compute_targets(episodes,critic,cfg.gamma,cfg.gae_lambda,bootstrap_truncation=True)
    states=torch.tensor(values['states'],dtype=torch.long)
    actions=torch.tensor(values['actions'],dtype=torch.long)
    old_logps=torch.tensor(values['old_logps'],dtype=torch.float64)
    old_values=torch.tensor(values['values_old'],dtype=torch.float64)
    returns=torch.tensor(values['lambda_returns'],dtype=torch.float64)
    raw=torch.tensor(values['advantages'],dtype=torch.float64)
    require(len(raw)>1,"全局优势归一化至少2步")
    normalized=(raw-raw.mean())/(raw.std(unbiased=False)+1e-8)
    with torch.no_grad():
        dist=F.log_softmax(actor(states),dim=-1)
        recalculated=dist.gather(-1,actions.unsqueeze(-1)).squeeze(-1)
    require(torch.allclose(old_logps,recalculated,atol=2e-6,rtol=2e-6),
            "采样时old_logp与当前Actor重算不一致")
    result={"states":states,"actions":actions,"old_logps":old_logps,
            "old_values":old_values,"returns":returns,
            "advantages":normalized.detach().clone(),"raw_advantages":raw,
            "old_distribution":dist.detach().clone(),
            "terminated":torch.tensor(values['terminated'],dtype=torch.bool),
            "truncated":torch.tensor(values['truncated'],dtype=torch.bool),
            "rollout_id":rollout_id,"sampled_steps":len(raw),
            "episode_lengths":[len(ep) for ep in episodes],
            "episode_successes":sum(ep[-1].terminated for ep in episodes),
            "episode_truncations":sum(ep[-1].truncated for ep in episodes),
            "actor_stopped":False,"closed":False,"finished_epochs":0,
            "actor_updates":0,"critic_updates":0,
            "actor_before_hash":tensor_hash(actor.state_dict()),
            "critic_before_hash":tensor_hash(critic.state_dict()),
            "expected_actor_hash":tensor_hash(actor.state_dict()),
            "expected_critic_hash":tensor_hash(critic.state_dict())}
    result['seal']=frozen_signature(result)
    return result


def verify_buffer(actor: Actor, critic: Critic,buffer: dict,cfg: Config,rollout_id: int):
    require(not buffer['closed'],"rollout已经关闭")
    require(buffer['rollout_id']==rollout_id,"不能跨rollout使用旧数据")
    require(buffer['seal']==frozen_signature(buffer),"冻结的rollout数组或元数据被篡改")
    require(buffer['finished_epochs']<cfg.epochs,"已超过计划epoch数")
    require(buffer['expected_actor_hash']==tensor_hash(actor.state_dict()),"Actor在受控循环外变化")
    require(buffer['expected_critic_hash']==tensor_hash(critic.state_dict()),"Critic在受控循环外变化")
    require(all(not buffer[n].requires_grad for n in (
        'old_logps','old_values','returns','advantages','old_distribution')),
        "旧策略、GAE与value target不得持有梯度图")

### 实现：minibatch与短尾microbatch的正确加权

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [8]:
def per_step_losses(actor: Actor,critic: Critic,buffer: dict,index: torch.Tensor,cfg: Config):
    require(index.dtype==torch.long and index.ndim==1 and index.numel()>0,"需要非空long索引")
    s=buffer['states'][index];a=buffer['actions'][index]
    old_lp=buffer['old_logps'][index]
    adv=buffer['advantages'][index].detach()
    old_v=buffer['old_values'][index].detach()
    target=buffer['returns'][index].detach()
    dist=F.log_softmax(actor(s),dim=-1)
    current_lp=dist.gather(-1,a.unsqueeze(-1)).squeeze(-1)
    ratio=probability_ratio(current_lp,old_lp)
    plain=ratio*adv
    clipped=ratio.clamp(1-cfg.clip_ratio,1+cfg.clip_ratio)*adv
    entropy=-(dist.exp()*dist).sum(-1)
    actor_per=-torch.minimum(plain,clipped)-cfg.entropy_coef*entropy
    predicted=critic(s)
    raw_square=(predicted-target).square()
    clipped_values=old_v+(predicted-old_v).clamp(-cfg.value_clip,cfg.value_clip)
    critic_per=.5*torch.maximum(raw_square,(clipped_values-target).square())
    active=((ratio>1+cfg.clip_ratio)&(adv>0))|((ratio<1-cfg.clip_ratio)&(adv<0))
    outside=(ratio>1+cfg.clip_ratio)|(ratio<1-cfg.clip_ratio)
    return actor_per,critic_per,{
        'ratio':ratio.detach(),'active_clip':active.detach(),'outside':outside.detach(),
        'entropy':entropy.detach(), 'old_minus_new':(old_lp-current_lp).detach(),
        'value_abs_error':(predicted-target).abs().detach()}


@torch.no_grad()
def rollout_diagnostics(actor: Actor,critic: Critic,buffer: dict,cfg: Config) -> dict:
    """整个已观察rollout按step频数统计，而不是平均3个状态或minibatch均值。"""
    actor.eval();critic.eval()
    dist=F.log_softmax(actor(buffer['states']),dim=-1)
    old=buffer['old_distribution'];a=buffer['actions']
    current_lp=dist.gather(-1,a[:,None]).squeeze(-1)
    ratio=probability_ratio(current_lp,buffer['old_logps'])
    old_minus_new=buffer['old_logps']-current_lp
    pointwise=(old.exp()*(old-dist)).sum(-1)
    require(float(pointwise.min())>-1e-8,"归一化分类KL数值异常")
    sample_estimator=((ratio-1)-ratio.log()).mean()  # 非负的近似估计形式
    adv=buffer['advantages']
    out=((ratio>1+cfg.clip_ratio)|(ratio<1-cfg.clip_ratio))
    active=((ratio>1+cfg.clip_ratio)&(adv>0))|((ratio<1-cfg.clip_ratio)&(adv<0))
    ent=-(dist.exp()*dist).sum(-1)
    predicted=critic(buffer['states'])
    result={'transitions':len(a),
        'old_state_kl_exact_visited':float(pointwise.mean()),
        'sampled_approx_kl':float(old_minus_new.mean()),
        'sampled_nonnegative_estimator':float(sample_estimator),
        'sampled_approx_may_be_negative':True,
        'ratio_min':float(ratio.min()), 'ratio_max':float(ratio.max()),
        'ratio_outside_fraction':float(out.double().mean()),
        'active_clip_fraction':float(active.double().mean()),
        'entropy_per_transition':float(ent.mean()),
        'critic_half_mse':float((.5*(predicted-buffer['returns']).square()).mean()),
        'mean_normalized_advantage':float(adv.mean()),
        'old_state_frequency':[int((buffer['states']==s).sum()) for s in range(GOAL)]}
    return result


def full_minibatch_gradients(actor: Actor,critic: Critic,buffer: dict,
                             minibatch: list[int],cfg: Config, microbatch_size: int,
                             compute_actor: bool=True):
    """通过一个采样minibatch中的microchunks累积；此函数不执行step。

    每个micro的loss按该micro行数求和后 / 当前minibatch总行数。
    对actor/critic分别反向，保证最后不满micro时也等价于直接full-minibatch求均值。
    """
    require(type(microbatch_size) is int and microbatch_size>0,'micro大小非法')
    require(bool(minibatch) and len(set(minibatch))==len(minibatch),"minibatch索引重复/空")
    require(all(type(i) is int and 0<=i<buffer['sampled_steps'] for i in minibatch),"索引越界")
    actor.zero_grad(set_to_none=True);critic.zero_grad(set_to_none=True)
    mini_size=len(minibatch)
    parts=split_microbatches(minibatch,microbatch_size)
    sums={'actor_loss_sum':0.,'critic_loss_sum':0.,'entropy_sum':0.,'clip_count':0,
          'outside_count':0,'old_minus_new_sum':0.,'abs_value_error_sum':0.}
    for micro in parts:
        index=torch.tensor(micro,dtype=torch.long)
        actor_per,critic_per,trace=per_step_losses(actor,critic,buffer,index,cfg)
        # Do NOT divide by len(parts) or by len(micro) here.
        if compute_actor:
            (actor_per.sum()/mini_size).backward()
        (critic_per.sum()/mini_size).backward()
        with torch.no_grad():
            sums['actor_loss_sum']+=float(actor_per.detach().sum())
            sums['critic_loss_sum']+=float(critic_per.detach().sum())
            sums['entropy_sum']+=float(trace['entropy'].sum())
            sums['clip_count']+=int(trace['active_clip'].sum())
            sums['outside_count']+=int(trace['outside'].sum())
            sums['old_minus_new_sum']+=float(trace['old_minus_new'].sum())
            sums['abs_value_error_sum']+=float(trace['value_abs_error'].sum())
    sums.update({'size':mini_size,'micro_sizes':[len(p) for p in parts],
                 'actor_loss_before':sums['actor_loss_sum']/mini_size,
                 'critic_loss_before':sums['critic_loss_sum']/mini_size,
                 'entropy_before':sums['entropy_sum']/mini_size,
                 'active_clip_fraction_before':sums['clip_count']/mini_size,
                 'ratio_outside_fraction_before':sums['outside_count']/mini_size,
                 'sampled_approx_kl_before':sums['old_minus_new_sum']/mini_size})
    return sums


def minibatch_update(actor: Actor,critic: Critic,buffer: dict,minibatch: list[int],
                     cfg: Config,actor_optimizer,critic_optimizer,microbatch_size: int) -> dict:
    require(bool(minibatch),"minibatch为空")
    old_diag=rollout_diagnostics(actor,critic,buffer,cfg)
    if not buffer['actor_stopped'] and old_diag['old_state_kl_exact_visited']>cfg.target_kl:
        buffer['actor_stopped']=True
    actor_update=not buffer['actor_stopped']
    actor_opt={id(p) for g in actor_optimizer.param_groups for p in g['params']}
    critic_opt={id(p) for g in critic_optimizer.param_groups for p in g['params']}
    require(actor_opt=={id(p) for p in actor.parameters()},"Actor优化器对象错误")
    require(critic_opt=={id(p) for p in critic.parameters()},"Critic优化器对象错误")
    require(not actor_opt&critic_opt,"Actor/Critic优化器不能共享对象")
    stats=full_minibatch_gradients(actor,critic,buffer,minibatch,cfg,microbatch_size,
                                   compute_actor=actor_update)
    if not actor_update:
        actor.zero_grad(set_to_none=True)
        actor_norm=None
    else:
        actor_norm=float(nn.utils.clip_grad_norm_(actor.parameters(),cfg.grad_clip,error_if_nonfinite=True))
        actor_optimizer.step()
        buffer['actor_updates']+=1
    critic_norm=float(nn.utils.clip_grad_norm_(critic.parameters(),cfg.grad_clip,error_if_nonfinite=True))
    critic_optimizer.step();buffer['critic_updates']+=1
    buffer['expected_actor_hash']=tensor_hash(actor.state_dict())
    buffer['expected_critic_hash']=tensor_hash(critic.state_dict())
    require(buffer['seal']==frozen_signature(buffer),"优化期间旧采样或目标被改变")
    return {'size':stats['size'],'micro_sizes':stats['micro_sizes'],
            'actor_updated':actor_update,'critic_updated':True,
            'actor_stopped':buffer['actor_stopped'],
            'old_state_kl_at_gate':old_diag['old_state_kl_exact_visited'],
            'sampled_kl_at_gate':old_diag['sampled_approx_kl'],
            'actor_grad_norm_before_clip':actor_norm,'critic_grad_norm_before_clip':critic_norm,
            **{k:v for k,v in stats.items() if k in ('actor_loss_before','critic_loss_before',
                'entropy_before','active_clip_fraction_before','ratio_outside_fraction_before',
                'sampled_approx_kl_before','clip_count','outside_count','actor_loss_sum',
                'critic_loss_sum','entropy_sum','old_minus_new_sum')}}

### 实现：独立训练器：每个训练seed从头开始

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [9]:
def summarise_preupdate_minibatches(rows: list[dict]) -> dict:
    """同一epoch所观察的minibatch，按有效transition总数求平均；短尾不获等权。"""
    require(isinstance(rows,list) and bool(rows),"至少一个minibatch")
    total=sum(r['size'] for r in rows)
    require(total>0,"没有transition")
    keys={'actor_loss':'actor_loss_sum', 'critic_loss':'critic_loss_sum',
          'entropy':'entropy_sum', 'sampled_approx_kl':'old_minus_new_sum'}
    result={k:sum(r[v] for r in rows)/total for k,v in keys.items()}
    result.update({'observed_steps':total, 'minibatches':len(rows),
                   'active_clip_fraction':sum(r['clip_count'] for r in rows)/total,
                   'ratio_outside_fraction':sum(r['outside_count'] for r in rows)/total,
                   'actor_updates':sum(r['actor_updated'] for r in rows),
                   'critic_updates':sum(r['critic_updated'] for r in rows),
                   'short_tail_size':rows[-1]['size'],
                   'note':'minibatch BEFORE-update metrics, weighted by actual transitions; not one simultaneous full-buffer evaluation'})
    return result


def train_arm(cfg: Config,mode: str,schedule: list[int],
              original_actor: Actor,original_critic: Critic):
    require(mode in {'full','mini','accum'},"未知训练对照")
    actor=copy.deepcopy(original_actor);critic=copy.deepcopy(original_critic)
    aopt=torch.optim.Adam(actor.parameters(),lr=cfg.actor_lr)
    vopt=torch.optim.Adam(critic.parameters(),lr=cfg.critic_lr)
    history=[];a_updates=c_updates=seen=total_episodes=stopped_rollouts=0
    for rid,seed in enumerate(schedule):
        eps=collect_rollouts(actor,cfg.episodes_per_rollout,cfg.horizon,random.Random(seed))
        buf=make_buffer(actor,critic,eps,cfg,rid)
        n=buf['sampled_steps']
        require(len(set(buf['episode_lengths']))>1 or cfg.episodes_per_rollout==1 or n>0,
                "回合长度异常")
        start_diag=rollout_diagnostics(actor,critic,buf,cfg)
        require(start_diag['old_state_kl_exact_visited']<1e-10,"新rollout的old state KL应为0")
        item={'rollout_id':rid,'episodes':cfg.episodes_per_rollout,'steps':n,
              'episode_lengths':buf['episode_lengths'],
              'terminations':buf['episode_successes'],'truncations':buf['episode_truncations'],
              'advantage_raw_mean':float(buf['raw_advantages'].mean()),
              'advantage_raw_std':float(buf['raw_advantages'].std(unbiased=False)),
              'advantages_normalized_once':True,
              'initial':start_diag,'epochs':[]}
        for epoch in range(cfg.epochs):
            mini_size=n if mode=='full' else cfg.minibatch_size
            order=shuffled_minibatches(n,mini_size,stable_epoch_seed(cfg.seed,rid,epoch))
            check_indices=[i for chunk in order for i in chunk]
            require(sorted(check_indices)==list(range(n)),"每epoch必须恰好覆盖所有transition")
            rows=[]
            for mini in order:
                micro_size=len(mini) if mode in {'full','mini'} else cfg.microbatch_size
                row=minibatch_update(actor,critic,buf,mini,cfg,aopt,vopt,micro_size)
                rows.append(row)
            aggregated=summarise_preupdate_minibatches(rows)
            require(aggregated['observed_steps']==n,"缺失transition或重复计算")
            item['epochs'].append({'epoch':epoch+1,'seed':stable_epoch_seed(cfg.seed,rid,epoch),
                                    'coverage_sha256':sha256(order),'batches':rows,
                                    'weighted':aggregated,
                                    'final_kl':rollout_diagnostics(actor,critic,buf,cfg)})
            buf['finished_epochs']+=1
            # 检查这一个epoch的后验状态仍满足被冻结的合同。
            require(buf['seal']==frozen_signature(buf),"GAE、回报或旧策略在epoch间变化")
        buf['closed']=True
        a_updates+=buf['actor_updates'];c_updates+=buf['critic_updates']
        stopped_rollouts+=int(buf['actor_stopped'])
        total_episodes+=cfg.episodes_per_rollout;seen+=n
        item['actor_updates']=buf['actor_updates'];item['critic_updates']=buf['critic_updates']
        item['actor_stopped']=buf['actor_stopped']
        if rid==0 or (rid+1)%8==0 or rid+1==len(schedule):
            dev=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,60,seed=6306)
            item['dev_observation']={'exact_success_within_limit':dev['exact_success_within_limit'],
                                     'value_rmse':dev['value_rmse']}
        history.append(item)
    return actor,critic,{'mode':mode,'history':history,'rollouts':len(schedule),
                         'episodes':total_episodes,'unique_sampled_steps':seen,
                         'actor_optimizer_steps':a_updates,'critic_optimizer_steps':c_updates,
                         'actor_stopped_rollouts':stopped_rollouts,
                         'transition_presentations':seen*cfg.epochs,
                         'selection':'fixed_final_update_no_test_selection',
                         'minibatch_rule':'full=single_step;mini/accum=17_transition_minis_with_5_transition_micros'}

## 3. 程序级“锁定计划”，不是事后修饰

`protocol.json` 固定任务、候选、train seed、holdout seed、eval seed规则、主要指标和bootstrap设置。修改计划字段就不再是同一个预定义实验，需要另建实验版本。

运行过程中先完整写出 `dev_runs.jsonl`，计算 `selection.json`，**读回并固定选择哈希**，才运行 holdout。完整报告和重新评分会从开发结果重新计算选择，核对不能在测试后悄悄变更。

这属于**代码内预先声明的协议**，不是某个外部平台认可的不可篡改预注册。公开教学源码里也看得见全部seed和任务，所以不能当作秘密测试集。

### 实现：不可静默修改的多seed实验计划与候选选择

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [10]:
DEFAULT_TASKS = [{"id": "horizon_5", "horizon": 5}, {"id": "horizon_7", "horizon": 7}]
DEFAULT_CANDIDATES = [{"id": "lr_low", "actor_lr": .004},
                      {"id": "lr_mid", "actor_lr": .012},
                      {"id": "lr_high", "actor_lr": .03}]


def make_protocol(quick: bool = False) -> dict:
    """教学预定义计划。quick是另一个明确标记的缩减实验，不与正式结果混称。"""
    require(type(quick) is bool, "quick必须bool")
    return {
        "version": VERSION, "experiment_id": "lesson64-dev-selection-holdout-seeds",
        "variant": "quick_smoke_not_main_experiment" if quick else "full_protocol",
        "tasks": copy.deepcopy(DEFAULT_TASKS),
        "candidates": copy.deepcopy(DEFAULT_CANDIDATES),
        "baseline_candidate": "lr_low",
        "development_seeds": [6401, 6402] if quick else [6401, 6402, 6403],
        "holdout_seeds": [6501, 6502] if quick else [6501, 6502, 6503, 6504],
        "training": {"method": "mini", "rollouts": 2 if quick else 6,
                     "episodes_per_rollout": 8 if quick else 12,
                     "epochs": 2, "minibatch_size": 17, "microbatch_size": 5,
                     "critic_lr": .010, "entropy_coef": .006,
                     "gae_lambda": .95, "gamma": .95,
                     "clip_ratio": .2, "value_clip": .2, "target_kl": .08,
                     "actor_hidden": 24, "critic_hidden": 24, "grad_clip": 1.,
                     "eval_episodes": 96},
        "development_selection": "mean exact 5/7-step success over dev seeds within task; equal task average; earliest candidate wins ties",
        "holdout_rule": "train from scratch with disjoint seeds; evaluate only predeclared baseline and dev-selected candidate",
        "paired_evaluation_seed": "910000 + 10 * training_seed + horizon",
        "bootstrap": {"confidence": .90, "resamples": 2000, "seed": 640064},
        "primary_metric": "analytically exact success probability within fixed external horizon, higher is better",
        "other_metrics": ["sampled_success_rate", "critic_value_rmse", "updates", "environment_transitions"],
        "results_interpretation": "percentile cluster bootstrap conditional on fixed tasks, not proof of nominal coverage with 4 runs",
        "leakage_boundary": "no holdout-trained model or score is available before dev selection is sealed",
    }


def validate_protocol(protocol: dict) -> None:
    require(type(protocol) is dict, "protocol类型错误")
    require(protocol.get("variant") in {"full_protocol", "quick_smoke_not_main_experiment"},
            "未知协议版本")
    reference = make_protocol(protocol["variant"] == "quick_smoke_not_main_experiment")
    require(canonical(protocol) == canonical(reference), "实验计划/种子/候选/预算被改动，不能偷偷重用实验ID")


def config_for(protocol: dict, task: dict, candidate: dict, seed: int) -> Config:
    validate_protocol(protocol)
    require(task in protocol["tasks"] and candidate in protocol["candidates"], "不属于协议的任务或候选")
    require(type(seed) is int and seed in protocol["development_seeds"] + protocol["holdout_seeds"],
            "训练种子不属于计划")
    t = protocol["training"]
    cfg = Config(seed=seed, gamma=t["gamma"], gae_lambda=t["gae_lambda"], horizon=task["horizon"],
                 rollouts=t["rollouts"], episodes_per_rollout=t["episodes_per_rollout"],
                 epochs=t["epochs"], minibatch_size=t["minibatch_size"],
                 microbatch_size=t["microbatch_size"], actor_lr=candidate["actor_lr"],
                 critic_lr=t["critic_lr"], actor_hidden=t["actor_hidden"],
                 critic_hidden=t["critic_hidden"], clip_ratio=t["clip_ratio"],
                 value_clip=t["value_clip"], entropy_coef=t["entropy_coef"],
                 target_kl=t["target_kl"], grad_clip=t["grad_clip"],
                 eval_episodes=t["eval_episodes"])
    cfg.validate()
    return cfg


def trial_key(row: dict) -> tuple[str, int, str]:
    return row["task_id"], row["seed"], row["candidate_id"]


def planned_development_keys(protocol: dict) -> set[tuple[str, int, str]]:
    return {(t["id"], seed, c["id"])
            for t in protocol["tasks"] for seed in protocol["development_seeds"]
            for c in protocol["candidates"]}


def planned_holdout_keys(protocol: dict, selected: str) -> set[tuple[str, int, str]]:
    ids = {protocol["baseline_candidate"], selected}
    return {(t["id"], seed, ident) for t in protocol["tasks"]
            for seed in protocol["holdout_seeds"] for ident in ids}


def index_trials(protocol: dict, rows: list[dict], phase: str, selected: str | None = None) -> dict:
    """所有计划位置必须齐全；失败实验不能静默删掉、补种子或平均幸存者。"""
    validate_protocol(protocol)
    require(phase in {"development", "holdout"} and type(rows) is list, "phase/records错误")
    allowed = (planned_development_keys(protocol) if phase == "development"
               else planned_holdout_keys(protocol, selected))
    result = {}
    for row in rows:
        require(type(row) is dict and row.get("phase") == phase, "跨阶段使用评分")
        key = trial_key(row)
        require(key in allowed and key not in result, "未知/重复/跨种子试验结果")
        require(row.get("status") == "completed", "缺失或失败结果需要明确报告，不能静默丢弃")
        metric = row.get("exact_success_within_limit")
        require(type(metric) in (int, float) and math.isfinite(metric) and 0 <= metric <= 1,
                "主要指标必须在0..1")
        require(type(row.get("sampled_successes")) is int and
                type(row.get("sampled_episodes")) is int and
                0 <= row["sampled_successes"] <= row["sampled_episodes"], "评测计数非法")
        result[key] = row
    require(set(result) == allowed, "计划位置未全部完成；不能把部分完成误报为全量")
    return result


def select_by_development(protocol: dict, dev_records: list[dict]) -> dict:
    """此函数没有测试结果参数；拒绝任何phase不为development的记录。"""
    index = index_trials(protocol, dev_records, "development")
    board = []
    for candidate in protocol["candidates"]:
        by_task = []
        for task in protocol["tasks"]:
            values = [index[task["id"], seed, candidate["id"]]["exact_success_within_limit"]
                      for seed in protocol["development_seeds"]]
            by_task.append({"task_id": task["id"], "seed_scores": values,
                            "mean": sum(values) / len(values)})
        board.append({"candidate_id": candidate["id"],
                      "score": sum(t["mean"] for t in by_task) / len(by_task),
                      "by_task": by_task})
    best = max(board, key=lambda r: r["score"])  # Python max稳定，平局取计划中最早项
    return {"selected_candidate": best["candidate_id"], "board": board,
            "selection_criterion": protocol["development_selection"],
            "phase": "development_only", "protocol_sha256": sha256(protocol)}

### 实现：统计底座：配对seed、IQM、分位数与分层bootstrap

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [11]:
def numeric(values, label: str, minimum: int = 1) -> list[float]:
    require(type(values) in (list, tuple) and len(values) >= minimum, label + "样本不足")
    require(all(type(x) in (int, float) and math.isfinite(x) for x in values), label+"须有限数值")
    return [float(v) for v in values]


def mean_std(values: list[float]) -> dict:
    xs = numeric(values, "seed_scores", minimum=2)
    avg = sum(xs) / len(xs)
    return {"n_training_seeds": len(xs), "mean": avg,
            "sample_std": math.sqrt(sum((x - avg)**2 for x in xs) / (len(xs)-1)),
            "minimum": min(xs), "maximum": max(xs), "sorted_scores": sorted(xs)}


def iqm(values: list[float]) -> float:
    """对经验分布中间50%的积分均值，支持n不是4的倍数；等权seed×task。"""
    xs = sorted(numeric(values, "IQM"))
    n = len(xs)
    total = 0.
    for i, value in enumerate(xs):
        left = max(i / n, .25)
        right = min((i + 1) / n, .75)
        total += value * max(0., right-left)
    return total / .5


def percentile(values: list[float], probability: float) -> float:
    xs = sorted(numeric(values, "bootstrap_statistics"))
    require(type(probability) in (int, float) and math.isfinite(probability)
            and 0 <= probability <= 1, "分位数范围非法")
    at = probability * (len(xs)-1)
    low = math.floor(at)
    high = math.ceil(at)
    return xs[low] + (xs[high]-xs[low]) * (at-low)


def paired_task_seed_differences(protocol: dict, records: list[dict],
                                 selected: str) -> list[dict]:
    index = index_trials(protocol, records, "holdout", selected)
    paired = []
    for task in protocol["tasks"]:
        for seed in protocol["holdout_seeds"]:
            baseline = index[task["id"], seed, protocol["baseline_candidate"]]
            chosen = index[task["id"], seed, selected]
            paired.append({"task_id": task["id"], "seed": seed,
                           "baseline": baseline["exact_success_within_limit"],
                           "selected": chosen["exact_success_within_limit"],
                           "difference": chosen["exact_success_within_limit"] - baseline["exact_success_within_limit"]})
    return paired


def stratified_paired_bootstrap(protocol: dict, paired: list[dict]) -> dict:
    """各任务内成对重采样训练seed，保留同一seed下两个算法的相关性。"""
    validate_protocol(protocol)
    by_task = {task["id"]: [] for task in protocol["tasks"]}
    expected = {(t["id"], s) for t in protocol["tasks"] for s in protocol["holdout_seeds"]}
    keys = set()
    for row in paired:
        key = (row["task_id"], row["seed"])
        require(key in expected and key not in keys, "配对seed错位、重复或未知")
        keys.add(key)
        for k in ("baseline", "selected", "difference"):
            require(type(row[k]) in (int, float) and math.isfinite(row[k]), "无效配对得分")
        require(abs(row["selected"]-row["baseline"]-row["difference"]) < 1e-12,
                "配对差值和原始分数不一致")
        by_task[row["task_id"]].append((row["seed"], row["difference"]))
    require(keys == expected, "配对位置缺失")
    for task in by_task:
        by_task[task].sort()  # 与磁盘行序无关，重采样结果完全可复现
    point = sum(sum(d for _, d in by_task[tid]) / len(by_task[tid]) for tid in by_task) / len(by_task)
    contract = protocol["bootstrap"]
    draws = []
    rng = random.Random(contract["seed"])
    for _ in range(contract["resamples"]):
        task_means = []
        for tid in by_task:
            diffs = [d for _, d in by_task[tid]]
            task_means.append(sum(diffs[rng.randrange(len(diffs))] for _ in diffs) / len(diffs))
        draws.append(sum(task_means)/len(task_means))
    alpha = 1 - contract["confidence"]
    return {"mean_paired_difference": point,
            "percentile_interval": [percentile(draws, alpha/2), percentile(draws, 1-alpha/2)],
            "confidence_level": contract["confidence"], "resamples": contract["resamples"],
            "seed": contract["seed"], "unit": "training_seed_paired_within_fixed_task",
            "scope": "conditional_on_two_fixed_tasks;_percentile_bootstrap_descriptive_with_four_test_seeds",
            "no_significance_claim": True}


def compare_heldout(protocol: dict, selection: dict, test_records: list[dict]) -> dict:
    require(selection.get("phase") == "development_only" and
            selection.get("protocol_sha256") == sha256(protocol), "选择信息与计划不符")
    selected = selection["selected_candidate"]
    require(selected in [c["id"] for c in protocol["candidates"]], "未登记的被选候选")
    index_trials(protocol, test_records, "holdout", selected)
    paired = paired_task_seed_differences(protocol, test_records, selected)
    by_task = []
    for task in protocol["tasks"]:
        rows = [row for row in paired if row["task_id"] == task["id"]]
        by_task.append({"task_id": task["id"],
                        "baseline": mean_std([r["baseline"] for r in rows]),
                        "selected": mean_std([r["selected"] for r in rows]),
                        "paired_difference": mean_std([r["difference"] for r in rows])})
    chosen = [r["selected"] for r in paired]
    baseline = [r["baseline"] for r in paired]
    return {"selected_id": selected, "baseline_id": protocol["baseline_candidate"],
            "paired_by_task_seed": paired, "by_task": by_task,
            "primary_selected_equal_task_mean": sum(x["selected"]["mean"] for x in by_task) / len(by_task),
            "primary_baseline_equal_task_mean": sum(x["baseline"]["mean"] for x in by_task) / len(by_task),
            "iqm_selected_pooled_fixed_tasks": iqm(chosen),
            "iqm_baseline_pooled_fixed_tasks": iqm(baseline),
            "bootstrap": stratified_paired_bootstrap(protocol, paired),
            "reporting_warning": "only 2 small tasks and 4 heldout training seeds; no general RL ranking or confidence in generalization"}

In [12]:
def sample_fake_development(protocol: dict, boost: str | None=None) -> list[dict]:
    rows=[]
    for task in protocol['tasks']:
        for seed in protocol['development_seeds']:
            for candidate in protocol['candidates']:
                score=.5+(0.1 if candidate['id']==boost else 0)
                rows.append({'phase':'development','status':'completed',
                  'task_id':task['id'],'seed':seed,'candidate_id':candidate['id'],
                  'exact_success_within_limit':score,
                  'sampled_successes':50,'sampled_episodes':100})
    return rows


def sample_fake_holdout(protocol: dict, selected: str) -> list[dict]:
    rows=[]
    for task in protocol['tasks']:
        for seed in protocol['holdout_seeds']:
            for candidate_id in {protocol['baseline_candidate'],selected}:
                value=.3 if candidate_id==protocol['baseline_candidate'] else .4
                rows.append({'phase':'holdout','status':'completed',
                    'task_id':task['id'],'seed':seed,'candidate_id':candidate_id,
                    'exact_success_within_limit':value,
                    'sampled_successes':30,'sampled_episodes':100})
    return rows

In [13]:
PROTOCOL = make_protocol()
print("候选学习率：",[(c["id"],c["actor_lr"]) for c in PROTOCOL["candidates"]])
print("开发训练seed：",PROTOCOL["development_seeds"])
print("留出训练seed：",PROTOCOL["holdout_seeds"])
print("开发运行总数：",len(planned_development_keys(PROTOCOL)))
print("若选中非基线，留出运行总数：",len(planned_holdout_keys(PROTOCOL,"lr_high")))
print("主要指标：",PROTOCOL["primary_metric"])

候选学习率： [('lr_low', 0.004), ('lr_mid', 0.012), ('lr_high', 0.03)]
开发训练seed： [6401, 6402, 6403]
留出训练seed： [6501, 6502, 6503, 6504]
开发运行总数： 18
若选中非基线，留出运行总数： 16
主要指标： analytically exact success probability within fixed external horizon, higher is better


## 4. 选择超参数：不能仅挑“最好看的seed”

假设三个候选在三个独立开发训练seed上各有分数。错误做法是拿每个候选的最高分比较、删去训练失败seed、反复查看留出seed并调整。

本课要求开发结果**全矩阵齐全**：每个任务×每个seed×每个候选恰好一份。缺失或重复都拒绝；status不为 completed 也不能默默从分母中删除。真实失败应记录失败阶段并重新规划实验，而不是事后把幸存者平均。

下面全部是**程序合成的单元测试记录**，不是训练结果。

In [14]:
fake_dev = sample_fake_development(PROTOCOL)
print("所有候选开发分相同时：",select_by_development(PROTOCOL,fake_dev)["selected_candidate"])
fake_advantaged = sample_fake_development(PROTOCOL,"lr_high")
print("给high更高开发分时：",select_by_development(PROTOCOL,fake_advantaged)["selected_candidate"])
try:
    select_by_development(PROTOCOL,fake_advantaged[:-1])
except LessonError as error:
    print("缺一个seed必须拒绝：",error)

所有候选开发分相同时： lr_low
给high更高开发分时： lr_high
缺一个seed必须拒绝： 计划位置未全部完成；不能把部分完成误报为全量


## 5. Winner's curse：同一指标调得越多，开发成绩可能越乐观

即使三个候选的真实质量完全相同，只要开发评估有噪声，选择“最大测量值”也倾向于选择幸运噪声。

下方模拟1000次：每次3个真实质量都为0.5的候选，开发测量有独立波动，然后选最高值，再做独立留出测量。**这是纯数学构造，不是本课 PPO 模型测出来的损失**。

调参空间扩大或反复盯着测试集时，风险更大。因此，测试结果必须只用来报告事先固定的选择，不能继续控制同一次选择。

### 实现：实验反例：赢家偏差与回合伪独立

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [15]:
def simple_percentile_bootstrap(values: list[float], seed: int = 440, reps: int = 2000) -> list[float]:
    xs = numeric(values, "bootstrap", minimum=2)
    rng = random.Random(seed)
    draws = [sum(xs[rng.randrange(len(xs))] for _ in xs)/len(xs) for _ in range(reps)]
    return [percentile(draws, .05), percentile(draws, .95)]


def statistical_counterexamples() -> dict:
    # 同一个训练策略产生100个高度相关(这里完全相同)评估结果。
    seed_scores = [0., 0., 1., 1.]
    episode_scores = [v for v in seed_scores for _ in range(100)]
    cluster_ci = simple_percentile_bootstrap(seed_scores, 441)
    pseudo_ci = simple_percentile_bootstrap(episode_scores, 441)
    # 所有候选真实质量相同，只是开发测量有独立噪声。
    rng = random.Random(64411)
    replications = 1000
    dev_winners, independent_tests = [], []
    for _ in range(replications):
        dev = [max(0., min(1., rng.gauss(.5, .12))) for _ in range(3)]
        chosen = max(range(3), key=lambda i: dev[i])
        dev_winners.append(dev[chosen])
        independent_tests.append(max(0., min(1., rng.gauss(.5, .12))))
    return {"pseudo_replication": {"independent_training_seeds": 4,
            "evaluation_episodes_per_seed": 100, "mean": .5,
            "seed_cluster_bootstrap_90pct": cluster_ci,
            "wrong_episode_iid_bootstrap_90pct": pseudo_ci,
            "note": "same 100 copies per seed is an extreme correlation counterexample; not real rollout scores"},
            "selection_bias_synthetic": {"equivalent_candidates": 3,
             "simulated_trials": replications,
             "mean_best_development_measurement": sum(dev_winners)/replications,
             "mean_independent_holdout_measurement": sum(independent_tests)/replications,
             "fixed_true_candidate_quality": .5,
             "note": "synthetic noise; not performance of PPO candidates"}}

In [16]:
COUNTEREXAMPLES=statistical_counterexamples()
print(json.dumps(COUNTEREXAMPLES["selection_bias_synthetic"],ensure_ascii=False,indent=2))

{
  "equivalent_candidates": 3,
  "simulated_trials": 1000,
  "mean_best_development_measurement": 0.5994849772201247,
  "mean_independent_holdout_measurement": 0.5052486287801723,
  "fixed_true_candidate_quality": 0.5,
  "note": "synthetic noise; not performance of PPO candidates"
}


## 6. 统计单位：为什么不能把 400 个评估回合当作 400 次训练？

构造4个训练seed的策略，每个策略产生100条**完全相同**的评估记录：两份全0、两份全1。

四份训练模型的平均成绩是0.5。如果把400条记录直接当独立样本，bootstrap区间会非常窄；若按四个训练seed重采样，区间明显更宽。

这是一个极端相关性反例，不是真实RL数据。但它说明：模型内的评估回合无法自动代替独立训练重启。([Henderson 等：Deep RL That Matters](https://ojs.aaai.org/index.php/AAAI/article/view/11694))

In [17]:
print(json.dumps(COUNTEREXAMPLES["pseudo_replication"],ensure_ascii=False,indent=2))

{
  "independent_training_seeds": 4,
  "evaluation_episodes_per_seed": 100,
  "mean": 0.5,
  "seed_cluster_bootstrap_90pct": [
    0.0,
    1.0
  ],
  "wrong_episode_iid_bootstrap_90pct": [
    0.4575,
    0.54
  ],
  "note": "same 100 copies per seed is an extreme correlation counterexample; not real rollout scores"
}


## 7. Holdout 要做“同种子成对比较”

两个配置在同一个训练seed上的差值：

\[
d_{t,s}=M(\text{selected},t,s)-M(\text{baseline},t,s)
\]

这里 t 是任务配置，s 是**训练seed**。每个任务内对这些差求平均，再对任务等权。相同训练seed用于两个配置，配对使初始化随机性更可比较；但不代表所有随机差异被消除了。

如果漏掉某个算法的一个seed，不能拿另一个seed顶替。即使两个算法的单独均值都算得出来，配对差值的方差也可能已经改变。记录的seed必须唯一、完整且与预登记集合一致。

In [18]:
fake_selected=select_by_development(PROTOCOL,sample_fake_development(PROTOCOL,"lr_high"))
fake_holdout=sample_fake_holdout(PROTOCOL,"lr_high")
fake_comparison=compare_heldout(PROTOCOL,fake_selected,fake_holdout)
print("合成常差值：",fake_comparison["bootstrap"])
print("IQM([0,1,2,3])=",iqm([0,1,2,3]))
print("IQM([0,0,0,1,100])=",iqm([0,0,0,1,100]),"；均值=",sum([0,0,0,1,100])/5)

合成常差值： {'mean_paired_difference': 0.10000000000000003, 'percentile_interval': [0.10000000000000003, 0.10000000000000003], 'confidence_level': 0.9, 'resamples': 2000, 'seed': 640064, 'unit': 'training_seed_paired_within_fixed_task', 'scope': 'conditional_on_two_fixed_tasks;_percentile_bootstrap_descriptive_with_four_test_seeds', 'no_significance_claim': True}
IQM([0,1,2,3])= 1.5
IQM([0,0,0,1,100])= 0.30000000000000004 ；均值= 20.2


## 8. 置信区间：条件、方向、局限必须写全

本课用**任务内配对训练seed的百分位bootstrap**：在每个任务配置内，有放回地抽取训练seed的**两算法成对差值**，先求任务均值，再对两个任务等权平均。

预登记：2000次重采样、固定伪随机种子、90%百分位区间。它描述当前两任务、四留出seed下，配对均值的抽样不确定性。**种子很少，不能保证名义90%覆盖，更不能把区间包含0当成算法一定相等。**

经典 RL 评估研究建议显示不确定性、分布概况与稳健汇总，如IQM；跨任务的分层bootstrap要符合任务与训练run的结构。[Agarwal等：Statistical Precipice](https://arxiv.org/abs/2108.13264) · [rliable](https://github.com/google-research/rliable)

本课两个horizon来自同一种玩具MDP，不主张外推到多个算法、自然语言任务或完整RL环境集合。

## 9. 除了平均值，还要保存分布与预算

报告每个任务的训练seed数量、均值、样本标准差、最小/最大值、按任务等权的平均结果，并补充中间50%样本的 IQM 与经验尾部分布。

训练预算不能只写“跑了6次rollout”而忽略数据用量：我们记录训练回合数、实际环境transition数、transition呈现次数、Actor与Critic optimizer.step次数，以及最终评估回合数。

Actor 因 KL 停止时，更新次数可能少于 Critic；策略不同，成功与截断时间也不同。所以“rollout次数一样”不等于“耗费时间、steps、FLOPs完全一样”。本课没有可靠硬件计时与GPU数据，不能声明哪个方案更省算力。

### 实现：独立run的随机初始化、状态追踪与可信重载

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [19]:
def trial_paths(out: Path, phase: str, task_id: str, candidate_id: str, seed: int) -> tuple[Path, Path]:
    require(phase in {"development", "holdout"}, "未知试验阶段")
    key = f"{phase}__{task_id}__{candidate_id}__seed{seed}"
    return out / "traces" / (key + ".json"), out / "weights" / (key + ".pt")


def run_trial(protocol: dict, task: dict, candidate: dict, seed: int, phase: str,
              output_dir: Path) -> dict:
    """单次试验从随机初始化开始；开发与测试训练种子互斥。"""
    cfg = config_for(protocol, task, candidate, seed)
    require((phase == "development" and seed in protocol["development_seeds"]) or
            (phase == "holdout" and seed in protocol["holdout_seeds"]), "阶段与种子不匹配")
    torch.set_num_threads(1)
    seed_all(seed)
    init_actor = Actor(cfg.actor_hidden).double()
    init_critic = Critic(cfg.critic_hidden).double()
    init_actor_hash, init_critic_hash = tensor_hash(init_actor.state_dict()), tensor_hash(init_critic.state_dict())
    rng = random.Random(seed + 1)
    schedule = [rng.randrange(2**31) for _ in range(cfg.rollouts)]
    actor, critic, trace = train_arm(cfg, protocol["training"]["method"], schedule, init_actor, init_critic)
    # 测试评分在dev选择前从来不会调用；同seed不同候选用同一个评估随机流。
    eval_seed = 910000 + 10 * seed + task["horizon"]
    evaluated = evaluate_policy(actor, critic, cfg.gamma, cfg.horizon,
                                cfg.eval_episodes, eval_seed)
    trace_file, weights_file = trial_paths(output_dir, phase, task["id"], candidate["id"], seed)
    trace_file.parent.mkdir(exist_ok=True, parents=True)
    weights_file.parent.mkdir(exist_ok=True, parents=True)
    trace_file.write_text(json.dumps(trace, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)+"\n", encoding="utf-8")
    state = {"actor": {k:v.detach().cpu().clone() for k,v in actor.state_dict().items()},
             "critic": {k:v.detach().cpu().clone() for k,v in critic.state_dict().items()}}
    torch.save({"version": VERSION, "phase": phase, "task": task,
                "candidate": candidate, "seed": seed, "cfg": asdict(cfg),
                "actor_initial_hash": init_actor_hash, "critic_initial_hash": init_critic_hash,
                "actor_state": state["actor"], "critic_state": state["critic"],
                "actor_hash": tensor_hash(state["actor"]), "critic_hash": tensor_hash(state["critic"])}, weights_file)
    return {"phase": phase, "status": "completed", "task_id": task["id"], "horizon": task["horizon"],
            "candidate_id": candidate["id"], "actor_lr": candidate["actor_lr"], "seed": seed,
            "exact_success_within_limit": evaluated["exact_success_within_limit"],
            "sampled_successes": evaluated["sampled_successes"],
            "sampled_episodes": evaluated["sampled_episodes"],
            "sampled_success_rate": evaluated["sampled_success_rate"],
            "critic_value_rmse": evaluated["value_rmse"],
            "training_episodes": trace["episodes"], "training_transitions": trace["unique_sampled_steps"],
            "transition_presentations": trace["transition_presentations"],
            "actor_optimizer_steps": trace["actor_optimizer_steps"],
            "critic_optimizer_steps": trace["critic_optimizer_steps"],
            "actor_early_stopped_rollouts": trace["actor_stopped_rollouts"],
            "schedule_sha256": sha256(schedule), "actor_initial_hash": init_actor_hash,
            "critic_initial_hash": init_critic_hash, "final_actor_hash": tensor_hash(state["actor"]),
            "final_critic_hash": tensor_hash(state["critic"]),
            "trace_sha256": sha256(trace),
            "trace_file": str(trace_file.relative_to(output_dir)),
            "weight_file": str(weights_file.relative_to(output_dir)),
            "eval_seed": eval_seed}


def validate_trials_against_files(folder: Path, protocol: dict, records: list[dict]) -> dict:
    """重评分前核对模型/训练轨迹的身份与摘要；不执行不可信代码或完整重训。"""
    validated = 0
    for row in records:
        task = next(t for t in protocol["tasks"] if t["id"] == row["task_id"])
        candidate = next(c for c in protocol["candidates"] if c["id"] == row["candidate_id"])
        cfg = config_for(protocol, task, candidate, row["seed"])
        trace_path, weights_path = trial_paths(folder, row["phase"], row["task_id"], row["candidate_id"], row["seed"])
        require(trace_path.is_file() and weights_path.is_file(), "缺少追溯轨迹或权重")
        require(trace_path.stat().st_size < 3_000_000 and weights_path.stat().st_size < 1_000_000,
                "超出教学报告大小限制")
        trace = json.loads(trace_path.read_text(encoding="utf-8"))
        require(sha256(trace) == row["trace_sha256"], "训练轨迹已变化")
        # 仅加载自己生成/可信来源的本地文件；weights_only并非任意文件安全认证。
        obj = torch.load(weights_path, map_location="cpu", weights_only=True)
        require(obj["version"] == VERSION and obj["cfg"] == asdict(cfg) and
                obj["candidate"] == candidate and obj["task"] == task and
                obj["seed"] == row["seed"] and obj["phase"] == row["phase"], "模型文件元数据不匹配")
        require(obj["actor_initial_hash"] == row["actor_initial_hash"] and
                obj["critic_initial_hash"] == row["critic_initial_hash"], "初始化指纹不匹配")
        require(tensor_hash(obj["actor_state"]) == row["final_actor_hash"] == obj["actor_hash"] and
                tensor_hash(obj["critic_state"]) == row["final_critic_hash"] == obj["critic_hash"],
                "权重摘要不匹配")
        actor = Actor(cfg.actor_hidden).double(); critic = Critic(cfg.critic_hidden).double()
        actor.load_state_dict(obj["actor_state"], strict=True)
        critic.load_state_dict(obj["critic_state"], strict=True)
        evaluated = evaluate_policy(actor, critic, cfg.gamma, cfg.horizon,
                                    cfg.eval_episodes, row["eval_seed"])
        require(abs(evaluated["exact_success_within_limit"]-row["exact_success_within_limit"]) < 1e-12 and
                evaluated["sampled_successes"] == row["sampled_successes"], "模型重载与评测记录不符")
        validated += 1
    return {"validated_runs": validated, "mode": "inference_recheck_not_training_resume",
            "checkpoint_trust_boundary": "load_only_files_generated_by_trusted_lesson"}

### 实现：开发锁定、holdout执行、报告构建与重评分

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [20]:
def save_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")


def save_jsonl(path: Path, rows: list[dict]) -> None:
    path.write_text("".join(canonical(row) + "\n" for row in rows), encoding="utf-8")


def read_json(path: Path, limit: int = 8_000_000):
    require(path.is_file() and path.stat().st_size <= limit, "报告缺失或过大")
    def unique(pairs):
        out = {}
        for key, value in pairs:
            require(key not in out, "重复JSON键")
            out[key] = value
        return out
    def fail(_):
        raise LessonError("NaN/Infinity不是合法JSON")
    return json.loads(path.read_text(encoding="utf-8"), object_pairs_hook=unique,
                      parse_constant=fail)


def read_jsonl(path: Path) -> list[dict]:
    require(path.is_file() and path.stat().st_size < 10_000_000, "JSONL缺失或过大")
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        require(line.strip(), "不允许空记录")
        rows.append(json.loads(line))
    return rows


def score_distribution_profile(values: list[float]) -> dict:
    xs = numeric(values, "profile")
    return {str(threshold): sum(x >= threshold for x in xs) / len(xs)
            for threshold in (.25, .5, .75, .9)}


def build_report(protocol: dict, dev_records: list[dict], selection: dict,
                 test_records: list[dict]) -> dict:
    validate_protocol(protocol)
    computed = select_by_development(protocol, dev_records)
    require(computed == selection, "选择不能在查看holdout后更改")
    comparison = compare_heldout(protocol, selection, test_records)
    selected = selection["selected_candidate"]
    baseline = protocol["baseline_candidate"]
    by_phase = {"development": dev_records, "holdout": test_records}
    budgets = {}
    for phase, rows in by_phase.items():
        budgets[phase] = {"runs": len(rows),
            "training_episodes": sum(r["training_episodes"] for r in rows),
            "unique_environment_transitions": sum(r["training_transitions"] for r in rows),
            "transition_presentations": sum(r["transition_presentations"] for r in rows),
            "actor_optimizer_steps": sum(r["actor_optimizer_steps"] for r in rows),
            "critic_optimizer_steps": sum(r["critic_optimizer_steps"] for r in rows),
            "final_evaluation_episodes": sum(r["sampled_episodes"] for r in rows),
            "warning": "Actual measured training actions, not GPU FLOPs, model tokens, or equal wall time"}
    pairs = comparison["paired_by_task_seed"]
    comparison["empirical_tail_profile"] = {
        "baseline": score_distribution_profile([r["baseline"] for r in pairs]),
        "selected": score_distribution_profile([r["selected"] for r in pairs])}
    return {"version": VERSION, "protocol_sha256": sha256(protocol),
            "selection_sha256": sha256(selection), "selected_candidate": selected,
            "predeclared_baseline": baseline, "selection": selection,
            "holdout": comparison, "budgets": budgets,
            "statistical_caveats": [
                "fixed synthetic 3-state environment with only two horizon variants",
                "development used 3 seeds/task/candidate; heldout uses 4 training seeds/task",
                "paired percentile cluster bootstrap does not ensure nominal 90% coverage with few seeds",
                "evaluation episodes are nested in a trained policy, not independent training runs",
                "selection on dev outcomes cannot be scored as independent generalization evidence",
                "fixed holdout seeds exposed in public teaching source; research protocol, not secret benchmark",
                "different PyTorch versions/devices may produce different weights despite identical seeds",
                "no pretrained LLM, no CUDA benchmark, no population-of-tasks generalization claim"],
            "baseline_and_selected_same": selected == baseline}


def new_output_dir(root: Path, variant: str) -> Path:
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
    out = root / (variant + "_" + stamp)
    out.mkdir(exist_ok=False)
    return out


def full_experiment(root="lesson_64_outputs", quick=False) -> Path:
    torch.set_num_threads(1)
    protocol = make_protocol(quick)
    validate_protocol(protocol)
    out = new_output_dir(Path(root), protocol["variant"])
    manifest = {"lesson": 64, "version": VERSION, "status": "running",
                "stage": "precommitted_protocol", "python": platform.python_version(),
                "torch": str(torch.__version__), "device": "cpu", "dtype": "float64",
                "protocol_sha256": sha256(protocol), "variant": protocol["variant"],
                "statistical_unit": "training_seed_clustered_within_task",
                "sources": SOURCES, "network_calls": False, "model_downloads": False,
                "not_a_pretrained_llm_benchmark": True}
    save_json(out / "manifest.json", manifest)
    save_json(out / "protocol.json", protocol)
    try:
        dev = []
        for task in protocol["tasks"]:
            for seed in protocol["development_seeds"]:
                for candidate in protocol["candidates"]:
                    print(f"DEV {task['id']} seed={seed} {candidate['id']}", flush=True)
                    trial = run_trial(protocol, task, candidate, seed, "development", out)
                    dev.append(trial)
                    with (out / "dev_runs.jsonl").open("a", encoding="utf-8") as stream:
                        stream.write(canonical(trial) + "\n")
        index_trials(protocol, dev, "development")
        selection = select_by_development(protocol, dev)
        save_json(out / "selection.json", selection)
        # 在执行任何holdout之前，必须从磁盘确认开发侧选择内容已固定。
        require(read_json(out / "selection.json") == selection, "选择文件读回失败")
        manifest["stage"] = "development_selection_sealed_before_holdout"
        manifest["selection_sha256"] = sha256(selection)
        save_json(out / "manifest.json", manifest)
        print("LOCKED selected candidate:", selection["selected_candidate"], flush=True)
        candidate_lookup = {c["id"]:c for c in protocol["candidates"]}
        selected_ids = [c["id"] for c in protocol["candidates"]
                        if c["id"] in {protocol["baseline_candidate"], selection["selected_candidate"]}]
        holdout = []
        for task in protocol["tasks"]:
            for seed in protocol["holdout_seeds"]:
                for candidate_id in selected_ids:
                    print(f"HOLDOUT {task['id']} seed={seed} {candidate_id}", flush=True)
                    row = run_trial(protocol, task, candidate_lookup[candidate_id], seed, "holdout", out)
                    holdout.append(row)
                    with (out / "holdout_runs.jsonl").open("a", encoding="utf-8") as stream:
                        stream.write(canonical(row) + "\n")
        index_trials(protocol, holdout, "holdout", selection["selected_candidate"])
        report = build_report(protocol, dev, selection, holdout)
        report["verification"] = {"development":validate_trials_against_files(out, protocol, dev),
                                  "holdout":validate_trials_against_files(out, protocol, holdout)}
        save_json(out / "report.json", report)
        save_json(out / "counterexamples.json", statistical_counterexamples())
        saved = rescore_run(out, validate_files=True, write_output=False)
        require(saved == report, "独立文件路径重评分与原报告不一致")
        manifest["stage"] = "heldout_completed_and_rechecked"
        manifest["status"] = "completed"
        save_json(out / "manifest.json", manifest)
        print("RESULT dev-selected:", selection["selected_candidate"],
              "heldout diff:", report["holdout"]["bootstrap"]["mean_paired_difference"], flush=True)
        print("OUTPUT:", out, flush=True)
        return out
    except Exception as exc:
        manifest["status"] = "failed"
        manifest["error_class"] = type(exc).__name__
        manifest["error_message"] = str(exc)[:300]
        save_json(out / "manifest.json", manifest)
        raise


def rescore_run(folder: str | Path, validate_files=True, write_output=True) -> dict:
    """只信任本课可信文件的原始运行记录，不直接信旧report的分数/选择。"""
    folder = Path(folder)
    protocol = read_json(folder / "protocol.json")
    validate_protocol(protocol)
    selection = read_json(folder / "selection.json")
    dev = read_jsonl(folder / "dev_runs.jsonl")
    test = read_jsonl(folder / "holdout_runs.jsonl")
    report = build_report(protocol, dev, selection, test)
    if validate_files:
        report["verification"] = {"development": validate_trials_against_files(folder, protocol, dev),
                                  "holdout": validate_trials_against_files(folder, protocol, test)}
    if write_output:
        out = new_output_dir(folder.parent, "rescore")
        save_json(out / "manifest.json", {"lesson": 64, "version": VERSION,
                  "mode": "read_only_source_recompute", "source_folder": str(folder),
                  "verification": bool(validate_files), "source_unchanged": True})
        save_json(out / "report.json", report)
        print("RESCORE:", out)
    return report

## 10. 完整实验：先在开发seed上选择，再运行留出seed

默认完整计划：两个horizon×三个候选×三个开发seed＝**18次开发训练**。选择以后，只在四个未见过的训练seed上运行事先声明的基线与选中候选，若不同则为**16次留出训练**。

每次run单独重新初始化 Actor 与 Critic，并使用相同的训练预算；训练参数记录、最终权重、轨迹和评估指标均保存。所有候选首先在开发阶段训练完毕后才决定选择对象。

如果初读想先跑一个较小完整闭环，可将 `USE_QUICK_PROTOCOL=True`。它会使用另一份明确标记的协议，不能与正式结果合并。**Notebook 默认执行正式完整计划**，全程无联网或模型下载。

In [21]:
RUN_FULL_EXPERIMENT=True
USE_QUICK_PROTOCOL=False
OUTPUT_ROOT=Path("lesson_64_outputs")
RUN_DIRECTORY=None
if RUN_FULL_EXPERIMENT:
    RUN_DIRECTORY=full_experiment(OUTPUT_ROOT,quick=USE_QUICK_PROTOCOL)
else:
    print("完整训练未运行：没有本次模型成绩，也不会伪造留出评估。")

DEV horizon_5 seed=6401 lr_low


DEV horizon_5 seed=6401 lr_mid


DEV horizon_5 seed=6401 lr_high


DEV horizon_5 seed=6402 lr_low


DEV horizon_5 seed=6402 lr_mid


DEV horizon_5 seed=6402 lr_high


... (27 outputs left out)

HOLDOUT horizon_7 seed=6504 lr_low


HOLDOUT horizon_7 seed=6504 lr_high


RESULT dev-selected: lr_high heldout diff: -0.0839701780760404


OUTPUT: lesson_64_outputs/full_protocol_run_20261103T085011_650955Z


## 11. 读取**本次执行结果**：开发排名与留出差

以下单元从当前运行文件中取值；若你跳过训练，就不显示虚构结果。

默认课程制作环境的CPU运行：开发阶段选中了 `lr_high`，但留出同seed配对平均差约为 −0.084；90%描述性bootstrap区间覆盖0。它说明单次开发最优未必在新训练seed上保持领先，不代表该算法普遍退化，也不能据这个小样本宣布统计显著。

**特别注意：这是事先固定的超参数候选与开发准则，不能看完留出就重新选 `lr_low` 并仍声称那是独立测试。**

In [22]:
FINAL_REPORT=None
if RUN_DIRECTORY is not None:
    FINAL_REPORT=read_json(RUN_DIRECTORY/"report.json")
    print("开发候选排名：")
    for row in FINAL_REPORT["selection"]["board"]:
        print(row["candidate_id"],"开发等权均值=",round(row["score"],7),
              "各任务均值=",[round(t["mean"],7) for t in row["by_task"]])
    print("开发选中：",FINAL_REPORT["selected_candidate"])
    test=FINAL_REPORT["holdout"]
    print("留出基线均值：",test["primary_baseline_equal_task_mean"])
    print("留出选中均值：",test["primary_selected_equal_task_mean"])
    print("配对差与90%区间：",test["bootstrap"])
else:
    print("没有本次留出报告。")

开发候选排名：
lr_low 开发等权均值= 0.9969775 各任务均值= [0.9939892, 0.9999657]
lr_mid 开发等权均值= 0.9964516 各任务均值= [0.9929421, 0.999961]
lr_high 开发等权均值= 0.9977086 各任务均值= [0.9965925, 0.9988246]
开发选中： lr_high
留出基线均值： 0.9989149182151871
留出选中均值： 0.9149447401391467
配对差与90%区间： {'confidence_level': 0.9, 'mean_paired_difference': -0.0839701780760404, 'no_significance_claim': True, 'percentile_interval': [-0.25416733842155886, 0.0016053801862670463], 'resamples': 2000, 'scope': 'conditional_on_two_fixed_tasks;_percentile_bootstrap_descriptive_with_four_test_seeds', 'seed': 640064, 'unit': 'training_seed_paired_within_fixed_task'}


In [23]:
if FINAL_REPORT is not None:
    print("按任务与训练seed的原始分数：")
    for row in FINAL_REPORT["holdout"]["paired_by_task_seed"]:
        print(row["task_id"],"seed=",row["seed"],
              "baseline=",round(row["baseline"],5),
              "selected=",round(row["selected"],5),
              "difference=",round(row["difference"],5))
    print("训练与评估工作量：")
    print(json.dumps(FINAL_REPORT["budgets"],ensure_ascii=False,indent=2))

按任务与训练seed的原始分数：
horizon_5 seed= 6501 baseline= 0.99776 selected= 1.0 difference= 0.00224
horizon_5 seed= 6502 baseline= 0.99964 selected= 1.0 difference= 0.00036
horizon_5 seed= 6503 baseline= 0.99905 selected= 0.31956 difference= -0.67949
horizon_5 seed= 6504 baseline= 0.99493 selected= 0.99999 difference= 0.00507
horizon_7 seed= 6501 baseline= 0.99995 selected= 1.0 difference= 5e-05
horizon_7 seed= 6502 baseline= 1.0 selected= 1.0 difference= 0.0
horizon_7 seed= 6503 baseline= 1.0 selected= 1.0 difference= 0.0
horizon_7 seed= 6504 baseline= 0.99999 selected= 1.0 difference= 1e-05
训练与评估工作量：
{
  "development": {
    "actor_optimizer_steps": 581,
    "critic_optimizer_steps": 714,
    "final_evaluation_episodes": 1728,
    "runs": 18,
    "training_episodes": 1296,
    "transition_presentations": 10160,
    "unique_environment_transitions": 5080,
    "warning": "Actual measured training actions, not GPU FLOPs, model tokens, or equal wall time"
  },
  "holdout": {
    "actor_optimizer_s

## 12. 磁盘重评分：不信任现有 report.json

`--rescore-run` 重新读取原始开发和留出run记录，**从开发结果重选超参数并核对选中对象**，逐模型重载检查权重摘要、配置、训练轨迹摘要和重评估指标。

它创建**新**报告文件，不修改原始输入。计算标识哈希不等于安全数字签名，`weights_only=True` 也不是任意不可信文件安全认证。**只加载你自己生成或经过核对的文件。**

我们不实现逐step精确断点续训：缺少原训练的优化器状态、所有RNG与训练游标；这里只验证已保存的策略及评估记录。

In [24]:
if RUN_DIRECTORY is not None:
    old_hashes={p.name:__import__('hashlib').sha256(p.read_bytes()).hexdigest()
                for p in RUN_DIRECTORY.iterdir() if p.is_file()}
    AGAIN=rescore_run(RUN_DIRECTORY,validate_files=True,write_output=True)
    print("重评分与原始结果一致：",AGAIN==FINAL_REPORT)
    print("源文件未被改写：",old_hashes=={
        p.name:__import__('hashlib').sha256(p.read_bytes()).hexdigest()
        for p in RUN_DIRECTORY.iterdir() if p.is_file()})

RESCORE: lesson_64_outputs/rescore_run_20261103T085017_361734Z
重评分与原始结果一致： True
源文件未被改写： True


## 13. 独立练习

请先写函数，再打开三个检查开关。检查器必须拒绝错误的seed配对、重复开发记录和不完整bootstrap输入。**不要用已见测试成绩决定开发候选。**

- **练习A：配对训练seed平均差。** 输入每行 seed、候选ID、score，要求同一seed两种配置都齐全，返回选中减基线的均值。
- **练习B：只在开发记录上选参数。** 三候选×两任务×多个seed，先任务内平均再任务等权，平分保留计划第一项，严格拒绝holdout行与缺失。
- **练习C：百分位bootstrap。** 单一任务的独立seed得分按固定seed与重复次数有放回采样，返回90%区间。该练习是单任务bootstrap的数学底座；完整比较仍使用上面**任务内配对**的分层函数。

In [25]:
RUN_EXERCISE_A=False
RUN_EXERCISE_B=False
RUN_EXERCISE_C=False

def paired_mean_ex(records, baseline_id, selected_id):
    raise NotImplementedError("逐seed配对，拒绝缺失/重复")

def choose_development_ex(candidates, dev, tasks, seeds):
    raise NotImplementedError("只能使用完整dev矩阵；平分取第一项")

def percentile_bootstrap_ex(values, seed=17, n_resamples=200, confidence=.90):
    raise NotImplementedError("对独立训练seed做有放回bootstrap")

In [26]:
def check_exercise_a(fn) -> dict:
    examples=[{"seed":1,"candidate_id":"base","score":.2},
              {"seed":1,"candidate_id":"new","score":.3},
              {"seed":2,"candidate_id":"base","score":.9},
              {"seed":2,"candidate_id":"new","score":.8}]
    require(abs(fn(examples,"base","new"))<1e-12, "逐seed差值应该抵消")
    require(abs(fn(list(reversed(examples)),"base","new"))<1e-12, "不能依赖记录顺序")
    for bad in (examples[:-1], examples+[examples[0]],
                [dict(row,score=float('nan')) if i==0 else row for i,row in enumerate(examples)]):
        try:fn(bad,"base","new")
        except (ValueError,TypeError):pass
        else:raise LessonError("应该拒绝不完整、重复或NaN的seed")
    return {"exercise":"A","passed":True}


def check_exercise_b(fn) -> dict:
    candidates=["first","second","third"]
    tasks=["h5","h7"];seeds=[1,2]
    rows=[]
    for t in tasks:
        for s in seeds:
            for cid in candidates:
                rows.append({"task":t,"seed":s,"candidate":cid,"phase":"dev",
                             "exact_success": .5})
    require(fn(candidates,rows,tasks,seeds)=="first", "平分必须保留预登记第一项")
    changed=[dict(row,exact_success=.6) if row['candidate']=='second' else row for row in rows]
    require(fn(candidates,changed,tasks,seeds)=="second", "应该选择开发集均值最高者")
    invalids=[rows[:-1],rows+[rows[0]], [dict(r,phase='holdout') if i==0 else r for i,r in enumerate(rows)]]
    for bad in invalids:
        try:fn(candidates,bad,tasks,seeds)
        except (ValueError,TypeError):pass
        else:raise LessonError("测试污染或缺失位置未拒绝")
    return {"exercise":"B","passed":True}


def check_exercise_c(fn) -> dict:
    require(fn([.2,.2,.2,.2],seed=44,n_resamples=100,confidence=.9)==[.2,.2],
            "常数的bootstrap应该退化成单点")
    a=fn([0.,1.,0.,1.],seed=44,n_resamples=200,confidence=.9)
    b=fn([0.,1.,0.,1.],seed=44,n_resamples=200,confidence=.9)
    require(a==b and 0<=a[0]<=a[1]<=1,"bootstrap应可复现且不越界")
    try:fn([.5],seed=44,n_resamples=200,confidence=.9)
    except (ValueError,TypeError):pass
    else:raise LessonError("单种子不应报告bootstrap CI")
    return {"exercise":"C","passed":True}

In [27]:
if RUN_EXERCISE_A: print("练习A:",check_exercise_a(paired_mean_ex))
if RUN_EXERCISE_B: print("练习B:",check_exercise_b(choose_development_ex))
if RUN_EXERCISE_C: print("练习C:",check_exercise_c(percentile_bootstrap_ex))
if not any((RUN_EXERCISE_A,RUN_EXERCISE_B,RUN_EXERCISE_C)):
    print("独立练习尚未完成，默认不开启；运行全部单元不代表完成练习。")

独立练习尚未完成，默认不开启；运行全部单元不代表完成练习。


## 14. 参考答案

请先完成自己的实现再展开查看。参考函数使用独立名称，不覆盖练习函数。

In [28]:
def paired_mean_reference(records: list[dict], baseline_id: str, selected_id: str) -> float:
    """输入是单任务的独立训练seed记录，必须两个算法逐seed成对。"""
    require(type(records) is list and len(records) >= 4, "至少两个训练种子、两个候选")
    lookup = {}
    for row in records:
        require(type(row) is dict and set(row) == {"seed", "candidate_id", "score"}, "配对行字段错误")
        require(type(row["seed"]) is int and type(row["candidate_id"]) is str and
                type(row["score"]) in (int,float) and math.isfinite(row["score"]), "行类型无效")
        key = row["seed"], row["candidate_id"]
        require(key not in lookup, "重复候选/seed")
        lookup[key] = float(row["score"])
    ids = {r["seed"] for r in records}
    require(baseline_id != selected_id and len(ids) >= 2 and
            set(lookup) == {(seed, cand) for seed in ids for cand in (baseline_id, selected_id)},
            "必须逐seed完整配对，不能将两组各自排序/过滤")
    return sum(lookup[(s,selected_id)] - lookup[(s,baseline_id)] for s in ids)/len(ids)


def candidate_select_reference(candidates: list[str], dev: list[dict],
                               tasks: list[str], seeds: list[int]) -> str:
    require(type(candidates) is list and len(candidates)>0 and
            len(set(candidates))==len(candidates), "候选不合法或重复")
    require(type(tasks) is list and tasks and len(set(tasks))==len(tasks), "任务不合法或重复")
    require(type(seeds) is list and len(seeds)>=2 and len(set(seeds))==len(seeds), "至少两个独立种子")
    require(type(dev) is list, "开发记录类型错误")
    expected={(t,s,c) for t in tasks for s in seeds for c in candidates}
    lookup={}
    for row in dev:
        require(type(row) is dict and set(row)=={"task","seed","candidate","phase","exact_success"},
                "开发记录字段不符")
        key=(row["task"],row["seed"],row["candidate"])
        require(row["phase"]=="dev" and key in expected and key not in lookup,
                "存在测试数据、重复或计划外的记录")
        v=row["exact_success"]
        require(type(v) in (int,float) and math.isfinite(v) and 0<=v<=1,
                "非有限或越界主要指标")
        lookup[key]=v
    require(set(lookup)==expected, "不能删除没跑成功的种子")
    by_candidate=[]
    for candidate in candidates:
        each_task=[sum(lookup[(t,s,candidate)] for s in seeds)/len(seeds) for t in tasks]
        by_candidate.append(sum(each_task)/len(each_task))
    return candidates[max(range(len(candidates)),key=lambda i:by_candidate[i])]


def bootstrap_interval_reference(values: list[float], seed: int=17,
                                 n_resamples: int=200, confidence: float=.90) -> list[float]:
    xs=numeric(values,"独立训练seed",minimum=2)
    require(type(seed) is int and seed>=0 and type(n_resamples) is int and n_resamples>=100,
            "seed/重采样次数非法")
    require(type(confidence) in (int,float) and math.isfinite(confidence)
            and 0<confidence<1,"置信水平非法")
    rng=random.Random(seed)
    means=[sum(xs[rng.randrange(len(xs))] for _ in xs)/len(xs) for _ in range(n_resamples)]
    alpha=(1-confidence)/2
    return [percentile(means,alpha),percentile(means,1-alpha)]

In [29]:
print("参考A:",check_exercise_a(paired_mean_reference))
print("参考B:",check_exercise_b(candidate_select_reference))
print("参考C:",check_exercise_c(bootstrap_interval_reference))

参考A: {'exercise': 'A', 'passed': True}
参考B: {'exercise': 'B', 'passed': True}
参考C: {'exercise': 'C', 'passed': True}


### 实现：完整固定检查与有针对性的故障检测

下面是本课自包含的实现单元，运行后再看短实验。展开源代码可逐行核查；不读取旁边的 `.py` 文件。

In [30]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def close(self,name,a,b,tol=1e-10):
        self.ok(name,abs(float(a)-float(b))<=tol)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,TypeError,KeyError):self.names.append(name);return
        raise AssertionError(name+":没有拒绝")


def sample_fake_development(protocol: dict, boost: str | None=None) -> list[dict]:
    rows=[]
    for task in protocol['tasks']:
        for seed in protocol['development_seeds']:
            for candidate in protocol['candidates']:
                score=.5+(0.1 if candidate['id']==boost else 0)
                rows.append({'phase':'development','status':'completed',
                  'task_id':task['id'],'seed':seed,'candidate_id':candidate['id'],
                  'exact_success_within_limit':score,
                  'sampled_successes':50,'sampled_episodes':100})
    return rows


def sample_fake_holdout(protocol: dict, selected: str) -> list[dict]:
    rows=[]
    for task in protocol['tasks']:
        for seed in protocol['holdout_seeds']:
            for candidate_id in {protocol['baseline_candidate'],selected}:
                value=.3 if candidate_id==protocol['baseline_candidate'] else .4
                rows.append({'phase':'holdout','status':'completed',
                    'task_id':task['id'],'seed':seed,'candidate_id':candidate_id,
                    'exact_success_within_limit':value,
                    'sampled_successes':30,'sampled_episodes':100})
    return rows


def run_checks() -> dict:
    torch.set_num_threads(1)
    c=Checks()
    full=make_protocol();quick=make_protocol(True)
    validate_protocol(full);validate_protocol(quick)
    c.ok('协议版本不同',full['variant']!=quick['variant'])
    c.ok('开发三训练seed测试四训练seed',len(full['development_seeds'])==3 and len(full['holdout_seeds'])==4)
    c.ok('两个任务三候选',len(full['tasks'])==2 and len(full['candidates'])==3)
    c.ok('开发测试种子完全分离',not set(full['development_seeds'])&set(full['holdout_seeds']))
    c.ok('开发设计完整18次',len(planned_development_keys(full))==18)
    c.ok('测试选择与基线共16次',len(planned_holdout_keys(full,'lr_high'))==16)
    c.ok('选中baseline时测试只运行8次',len(planned_holdout_keys(full,'lr_low'))==8)
    for field,value in [('bootstrap',{'seed':10}),('tasks',[{'id':'horizon_5','horizon':4}]),
                        ('development_seeds',[6401,6402,6501]),('training',{'rollouts':1})]:
        bad=copy.deepcopy(full);bad[field]=value
        c.rejects('不允许静默修改协议_'+field,lambda bad=bad:validate_protocol(bad))
    bad=copy.deepcopy(full);bad['training']['rollouts']=True
    c.rejects('bool不能冒充预登记整数',lambda:validate_protocol(bad))
    for task in full['tasks']:
        for seed in full['development_seeds']:
            first=config_for(full,task,full['candidates'][0],seed)
            middle=config_for(full,task,full['candidates'][1],seed)
            c.ok('固定预算只改变actor_lr_'+task['id']+str(seed),
                 {k:v for k,v in asdict(first).items() if k!='actor_lr'}==
                 {k:v for k,v in asdict(middle).items() if k!='actor_lr'})
    c.rejects('未知训练种子被拒绝',lambda:config_for(full,full['tasks'][0],full['candidates'][0],9999))
    c.rejects('未登记候选被拒绝',lambda:config_for(full,full['tasks'][0],{'id':'bad'},6401))
    rows=sample_fake_development(full)
    winner=select_by_development(full,rows)
    c.ok('开发全平分必须选第一候选',winner['selected_candidate']=='lr_low')
    c.ok('选择阶段独立于holdout',winner['phase']=='development_only')
    boosted=sample_fake_development(full,'lr_high')
    chosen=select_by_development(full,boosted)
    c.ok('开发更高者被选中',chosen['selected_candidate']=='lr_high')
    c.close('每任务等权而非episode加权',chosen['board'][2]['score'],.6)
    c.ok('开发行序不影响选择',select_by_development(full,list(reversed(boosted)))==chosen)
    c.rejects('缺失开发run拒绝',lambda:select_by_development(full,boosted[:-1]))
    c.rejects('重复开发run拒绝',lambda:select_by_development(full,boosted+[boosted[0]]))
    polluted=copy.deepcopy(boosted);polluted[0]['phase']='holdout'
    c.rejects('测试结果不得进入开发选择',lambda:select_by_development(full,polluted))
    bad=copy.deepcopy(boosted);bad[0]['exact_success_within_limit']=float('nan')
    c.rejects('NaN不进入最佳模型选择',lambda:select_by_development(full,bad))
    bad=copy.deepcopy(boosted);bad[0]['status']='failed'
    c.rejects('训练失败不能从种子均值删掉',lambda:select_by_development(full,bad))
    # 改动辅助指标但保持exact主要指标时，选择结果不能发生变化。
    altered=copy.deepcopy(boosted)
    for item in altered:
        item['sampled_successes']=100 if item['candidate_id']=='lr_low' else 0
    c.ok('不能用噪声采样成功偷换事先主要指标',
         select_by_development(full,altered)['selected_candidate']=='lr_high')
    test=sample_fake_holdout(full,'lr_high')
    result=compare_heldout(full,chosen,test)
    c.close('配对差值应当0.1',result['bootstrap']['mean_paired_difference'],.1)
    c.close('选中候选平均0.4',result['primary_selected_equal_task_mean'],.4)
    c.close('预登记基线平均0.3',result['primary_baseline_equal_task_mean'],.3)
    c.ok('所有常差值重采样CI不变',all(abs(v-.1)<1e-12 for v in result['bootstrap']['percentile_interval']))
    c.ok('配对行序不能影响CI',stratified_paired_bootstrap(full,list(reversed(result['paired_by_task_seed'])))==
         result['bootstrap'])
    c.ok('holdout报告包含任务和seed',len(result['paired_by_task_seed'])==8 and len(result['by_task'])==2)
    c.rejects('holdout缺失拒绝',lambda:compare_heldout(full,chosen,test[:-1]))
    c.rejects('holdout重复拒绝',lambda:compare_heldout(full,chosen,test+[test[0]]))
    bad=copy.deepcopy(test);bad[0]['seed']=6401
    c.rejects('holdout含开发seed拒绝',lambda:compare_heldout(full,chosen,bad))
    bad=copy.deepcopy(chosen);bad['selected_candidate']='lr_mid'
    c.rejects('holdout前确定的选择不能改',lambda:build_report(full,boosted,bad,test))
    bad=copy.deepcopy(test);bad[0]['exact_success_within_limit']=1.1
    c.rejects('非法holdout成功率拒绝',lambda:compare_heldout(full,chosen,bad))
    bad=copy.deepcopy(result['paired_by_task_seed']);bad[0]['difference']*=2
    c.rejects('配对差必须来自原始分数',lambda:stratified_paired_bootstrap(full,bad))
    bad=copy.deepcopy(result['paired_by_task_seed']);bad[0]['seed']=6502
    c.rejects('不能重复seed配对',lambda:stratified_paired_bootstrap(full,bad))
    c.close('iqm四个点取中间一半',iqm([0,1,2,3]),1.5)
    c.close('iqm分段权重适用五个点',iqm([0,0,0,1,100]),.3)
    c.close('均值受异常值影响',sum([0,0,0,1,100])/5,20.2)
    c.close('单一值iqm正常定义',iqm([.7]),.7)
    c.close('分位数线性插值',percentile([0,10],.25),2.5)
    c.rejects('iqm不接受NaN',lambda:iqm([.1,float('nan')]))
    c.rejects('样本标准差至少两seed',lambda:mean_std([.5]))
    c.close('样本标准差分母n-1',mean_std([0,2])['sample_std'],math.sqrt(2))
    c.ok('种子分布完整报告',mean_std([0,2])['sorted_scores']==[0,2])
    counter=statistical_counterexamples()
    cluster=counter['pseudo_replication']['seed_cluster_bootstrap_90pct']
    naive=counter['pseudo_replication']['wrong_episode_iid_bootstrap_90pct']
    c.ok('同策略episode当独立样本会虚窄CI',cluster[1]-cluster[0] > 5*(naive[1]-naive[0]))
    c.ok('合成赢家开发分数高于独立测试',counter['selection_bias_synthetic']['mean_best_development_measurement']>
         counter['selection_bias_synthetic']['mean_independent_holdout_measurement']+.03)
    c.ok('标准区间不是无条件显著性声明',result['bootstrap']['no_significance_claim'])
    # 同一seed重启、同一任务、同一配置：CPU double和单线程应可逐位复现。
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        tiny=make_protocol(True);task=tiny['tasks'][0];cand=tiny['candidates'][1]
        a=run_trial(tiny,task,cand,6401,'development',Path(tmp)/'one')
        b=run_trial(tiny,task,cand,6401,'development',Path(tmp)/'two')
        for field in ('exact_success_within_limit','sampled_successes','critic_value_rmse',
                      'actor_initial_hash','critic_initial_hash','final_actor_hash','final_critic_hash',
                      'trace_sha256','schedule_sha256','training_transitions','actor_optimizer_steps'):
            c.ok('重复相同训练seed_'+field,a[field]==b[field])
        d=run_trial(tiny,task,cand,6402,'development',Path(tmp)/'three')
        c.ok('不同训练seed初始化不同',a['actor_initial_hash']!=d['actor_initial_hash'])
        c.ok('同一seed跨候选初始化相同',
             a['actor_initial_hash']==run_trial(tiny,task,tiny['candidates'][0],6401,
                                                 'development',Path(tmp)/'four')['actor_initial_hash'])
        verify=validate_trials_against_files(Path(tmp)/'one',tiny,[a])
        c.ok('可信文件可独立重载核对',verify['validated_runs']==1)
        tracepath=Path(tmp)/'one'/a['trace_file']
        trace=read_json(tracepath);trace['rollouts']+=1
        save_json(tracepath,trace)
        c.rejects('训练轨迹篡改会被发现',lambda:validate_trials_against_files(Path(tmp)/'one',tiny,[a]))
    c.ok('练习A参考',check_exercise_a(paired_mean_reference)['passed'])
    c.ok('练习B参考',check_exercise_b(candidate_select_reference)['passed'])
    c.ok('练习C参考',check_exercise_c(bootstrap_interval_reference)['passed'])
    return {"count":len(c.names),"names":c.names,
            "scope":"CPU双精度三状态小实验；独立训练seed的统计检查，不是LLM或RL普适排序"}


def mutation_checks() -> dict:
    """用真正会失败的参考检查器侦测常见统计错误。"""
    cases=[]
    def detect(name,action):
        try:action()
        except (AssertionError,LessonError,ValueError):cases.append({"fault":name,"detected":True})
        else:raise AssertionError("原验收没有检出错误："+name)
    def wrong_unpaired(rows,base,new):
        by_base=sorted([r['score'] for r in rows if r['candidate_id']==base])
        by_new=sorted([r['score'] for r in rows if r['candidate_id']==new])
        return sum(b-a for a,b in zip(by_base,by_new))/len(by_base)
    # 对均值本身，独立排序后再相减仍会给同一结果，故故障场景应检查配对结构。
    def wrong_missing(rows,base,new):
        rows=[r for r in rows if r['seed']!=2]
        return paired_mean_reference(rows,base,new)
    detect('静默删除缺失训练种子',lambda:check_exercise_a(wrong_missing))
    def wrong_tie(candidates,dev,tasks,seeds):return candidates[-1]
    detect('开发平局取最后候选',lambda:check_exercise_b(wrong_tie))
    def wrong_test_leak(candidates,dev,tasks,seeds):
        return 'second' if any(r['phase']=='holdout' for r in dev) else candidate_select_reference(candidates,dev,tasks,seeds)
    detect('把测试阶段结果送进选择器',lambda:check_exercise_b(wrong_test_leak))
    def wrong_bootstrap(values,seed=17,n_resamples=200,confidence=.9):
        return [sum(values)/len(values)]*2
    detect('把均值当成置信区间',lambda:check_exercise_c(wrong_bootstrap))
    def wrong_iqm(xs):return sum(xs)/len(xs)
    detect('把均值冒充IQM',lambda:Checks().close('IQM',wrong_iqm([0,0,0,1,100]),.3))
    def wrong_episode_ci():
        x=statistical_counterexamples()['pseudo_replication']
        c=Checks();c.ok('seed cluster必须比episode iid更宽',
                       x['wrong_episode_iid_bootstrap_90pct'][1]-x['wrong_episode_iid_bootstrap_90pct'][0]>
                       5*(x['seed_cluster_bootstrap_90pct'][1]-x['seed_cluster_bootstrap_90pct'][0]))
    detect('把400个episode当400个独立训练run',wrong_episode_ci)
    def wrong_select_test():
        protocol=make_protocol()
        dev=sample_fake_development(protocol,'lr_high')
        selection=select_by_development(protocol,dev)
        bad=copy.deepcopy(selection);bad['selected_candidate']='lr_mid'
        build_report(protocol,dev,bad,sample_fake_holdout(protocol,'lr_high'))
    detect('看过holdout后改换开发选项',wrong_select_test)
    return {"count":len(cases),"cases":cases,"scope":"targeted_protocol_mutations_not_formal_proof"}

In [31]:
CHECKS=run_checks()
MUTATIONS=mutation_checks()
print("固定检查通过：",CHECKS["count"])
print("故障检测：",MUTATIONS["count"])
for item in MUTATIONS["cases"]:
    print(item["fault"],"→",item["detected"])
if RUN_DIRECTORY is not None:
    save_json(RUN_DIRECTORY/"checks.json",CHECKS)
    save_json(RUN_DIRECTORY/"mutations.json",MUTATIONS)

固定检查通过： 74
故障检测： 7
静默删除缺失训练种子 → True
开发平局取最后候选 → True
把测试阶段结果送进选择器 → True
把均值当成置信区间 → True
把均值冒充IQM → True
把400个episode当400个独立训练run → True
看过holdout后改换开发选项 → True


## 15. CLI 与最终验收

配套脚本不依赖 Notebook，运行目录不会覆盖已有结果：

```bash
python lesson_64_ppo_multiseed_reproducibility_protocol.py --checks-only
python lesson_64_ppo_multiseed_reproducibility_protocol.py --math-only
python lesson_64_ppo_multiseed_reproducibility_protocol.py
python lesson_64_ppo_multiseed_reproducibility_protocol.py --quick
python lesson_64_ppo_multiseed_reproducibility_protocol.py --rescore-run lesson_64_outputs/实际full_protocol目录
```

**验收问题：**

1. 为什么“评估400个回合”不能写成“400次独立训练”？
2. 为什么要锁定开发候选，再训练新的留出seed，而不是只换评估回合seed？
3. bootstrap为什么对同一任务下相同seed的**两种算法分数成对重采样**？
4. 为什么结果区间含0，不能宣布两个算法必然相等？
5. 为什么 `lr_high` 的开发结果稍好，却可能在留出训练seed上不如低学习率？
6. 三个候选、两种horizon、四留出seed的结论，为何不能推广到预训练模型或RL算法总体？
7. 环境transition次数和Actor/Critic优化器次数不完全一致时，如何报告实际工作量？

**下一课第65课：实验消融、故障归因与训练报告复核。** 保留多seed协议，研究怎样拆开GAE、value clipping、KL停止与batching效应，而不是同时改变所有超参数。

## 参考与诚实边界

- [Agarwal 等：Deep Reinforcement Learning at the Edge of the Statistical Precipice](https://arxiv.org/abs/2108.13264)：少种子评测、IQM、分层bootstrap与性能分布。
- [google-research/rliable](https://github.com/google-research/rliable)：可复查的RL可靠评测工具，本课未安装、未声称使用它。
- [Henderson 等：Deep Reinforcement Learning That Matters](https://ojs.aaai.org/index.php/AAAI/article/view/11694)：复现、随机性与报告问题。
- [Stable-Baselines3 RL Tips](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html)：使用多个训练seed与独立评估。
- [PPO 原论文](https://arxiv.org/abs/1707.06347)、[Spinning Up PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)：本课内置训练器的机制背景。
- [PyTorch可复现性](https://docs.pytorch.org/docs/stable/notes/randomness.html)：seed固定不保证跨软件/硬件逐位一致。

本课默认只对双精度CPU三状态教学MDP做重复试验；两种horizon不代表多种无关任务，开发与测试种子对教学源码可见，不能称作秘密基准。少种子百分位区间仅提供描述性不确定性，不作正式显著性断言；没有自然语言模型、CUDA或RTX3060性能数据。